# Bangalore Cab Fare Prediction — Baseline Modeling

This notebook builds the first machine learning models for predicting Uber cab fares.

The workflow is:

1. Load and clean the modeling dataset
2. Select the final features
3. Define features (`X`) and target (`y`)
4. Create a train / cross-validation / test strategy
5. Build the preprocessing pipeline
6. Establish a baseline model
7. Train stronger models
8. Compare regression metrics
9. Analyze model errors
10. Improve the model

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 1. Load the Processed Fare Dataset

The processed dataset contains the Uber fare observations along with the route-level features required for modeling.

In [2]:
DATA_PATH = "../data/processed/fares/uber_fares_processed.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (27420, 13)


,timestamp,time_slot,day_of_week,pickup,dropoff,pickup_category,drop_category,road_distance_km,travel_time_minutes,ride_name,seats,eta_minutes,current_price
0,2026-08-30 16:22:06.459051+05:30,15:00,Sunday,Brand Factory,NCS Elite,mall,residential_area,23.757,50.42,Go Non AC,4.0,5.0,528.89
1,2026-08-30 16:22:06.459051+05:30,15:00,Sunday,Brand Factory,NCS Elite,mall,residential_area,23.757,50.42,Auto,3.0,2.0,533.30
2,2026-08-30 16:22:06.459051+05:30,15:00,Sunday,Brand Factory,NCS Elite,mall,residential_area,23.757,50.42,Uber Go AC,4.0,4.0,609.13
3,2026-08-30 16:22:06.459051+05:30,15:00,Sunday,Brand Factory,NCS Elite,mall,residential_area,23.757,50.42,Bike Saver,1.0,7.0,324.58
4,2026-08-30 16:22:06.459051+05:30,15:00,Sunday,Brand Factory,NCS Elite,mall,residential_area,23.757,50.42,Premier AC,4.0,4.0,793.81


In [3]:
print("Zero-fare observations:", (df["current_price"] == 0).sum())

Zero-fare observations: 1


In [4]:
df = df[df["current_price"] > 0].copy()

print("Dataset shape after removing zero-fare observation:", df.shape)
print("Minimum fare:", df["current_price"].min())

Dataset shape after removing zero-fare observation: (27419, 13)
Minimum fare: 4.08


In [11]:
# Inspect the minimum-fare observation(s)

min_fare = df["current_price"].min()

print("Minimum fare:", min_fare)
print("Number of observations with minimum fare:", (df["current_price"] == min_fare).sum())

print("\n" + "="*80)
print("MINIMUM-FARE OBSERVATION(S)")
print("="*80)

min_fare_rows = df[df["current_price"] == min_fare].copy()

display(min_fare_rows)

print("\n" + "="*80)
print("DETAILS OF MINIMUM-FARE OBSERVATION(S)")
print("="*80)

for idx, row in min_fare_rows.iterrows():
    print(f"\nIndex: {idx}")
    print(f"Timestamp: {row['timestamp']}")
    print(f"Time slot: {row['time_slot']}")
    print(f"Day: {row['day_of_week']}")
    print(f"Pickup: {row['pickup']}")
    print(f"Dropoff: {row['dropoff']}")
    print(f"Pickup category: {row['pickup_category']}")
    print(f"Drop category: {row['drop_category']}")
    print(f"Ride name: {row['ride_name']}")
    print(f"Road distance (km): {row['road_distance_km']}")
    print(f"Travel time (minutes): {row['travel_time_minutes']}")
    print(f"Seats: {row['seats']}")
    print(f"ETA (minutes): {row['eta_minutes']}")
    print(f"Current price: ₹{row['current_price']}")
    
print("\n" + "="*80)
print("TARGET DISTRIBUTION AROUND THE MINIMUM")
print("="*80)

print("\n10 smallest fares:")
display(
    df[["timestamp", "time_slot", "day_of_week",
        "pickup", "dropoff", "ride_name", "current_price"]]
    .sort_values("current_price")
    .head(10)
)

print("\nFare statistics:")
display(df["current_price"].describe())

print("\n" + "="*80)
print("SAME ROUTE — OTHER OBSERVATIONS")
print("="*80)

for _, row in min_fare_rows.iterrows():
    same_route = df[
        (df["pickup"] == row["pickup"]) &
        (df["dropoff"] == row["dropoff"])
    ].sort_values("timestamp")
    
    print(f"\nRoute: {row['pickup']} → {row['dropoff']}")
    print("Number of observations for this route:", len(same_route))
    
    display(
        same_route[
            ["timestamp", "time_slot", "day_of_week",
             "ride_name", "road_distance_km",
             "travel_time_minutes", "current_price"]
        ].head(20)
    )

Minimum fare: 4.08
Number of observations with minimum fare: 1

MINIMUM-FARE OBSERVATION(S)


,timestamp,time_slot,day_of_week,pickup,dropoff,pickup_category,drop_category,road_distance_km,travel_time_minutes,ride_name,seats,eta_minutes,current_price
18676,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Canopy Citadel,Tuscan Infinite,residential_area,residential_area,3.154,8.75,Auto,3.0,2.0,4.08



DETAILS OF MINIMUM-FARE OBSERVATION(S)

Index: 18676
Timestamp: 2026-09-18 07:26:42.117985+05:30
Time slot: 00:00
Day: Friday
Pickup: Canopy Citadel
Dropoff: Tuscan Infinite
Pickup category: residential_area
Drop category: residential_area
Ride name: Auto
Road distance (km): 3.154
Travel time (minutes): 8.75
Seats: 3.0
ETA (minutes): 2.0
Current price: ₹4.08

TARGET DISTRIBUTION AROUND THE MINIMUM

10 smallest fares:


,timestamp,time_slot,day_of_week,pickup,dropoff,ride_name,current_price
18676,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Canopy Citadel,Tuscan Infinite,Auto,4.08
27310,2026-09-20 06:06:55.065200+05:30,00:00,Sunday,Cubbon Park,Government Maternity Hospital,Auto,10.12
9517,2026-09-16 15:25:18.658539+05:30,15:00,Wednesday,Nalapad's Hotel Bangalore International,Bangalore Cantonment,Auto,10.98
18224,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Peninsular Gneiss National Monument,Garuda Mall,Auto,13.28
4870,2026-09-15 09:00:09.535556+05:30,09:00,Tuesday,Mallya Audikesh Diagnostics Center,Nalapad's Hotel Bangalore International,Auto,14.66
19003,2026-09-18 07:26:42.117985+05:30,00:00,Friday,International Business School,Huskur Road,Auto,18.41
9761,2026-09-16 21:03:40.764861+05:30,21:00,Wednesday,Hostel Office,Srirampura,Auto,19.55
6694,2026-09-15 22:56:31.657930+05:30,21:00,Tuesday,Kamakshi Silk,VRL Bus Station,Auto,21.24
4400,2026-09-15 08:45:38.296053+05:30,00:00,Tuesday,Lalbagh Nursing Home,BMTC Hospital,Auto,21.33
10063,2026-09-16 21:13:15.890024+05:30,21:00,Wednesday,Srirampura,Amenities Hall,Auto,22.02



Fare statistics:


count    27419.000000
mean       632.030906
std        465.478851
min          4.080000
25%        284.245000
50%        514.100000
75%        873.300000
max       3987.610000
Name: current_price, dtype: float64


SAME ROUTE — OTHER OBSERVATIONS

Route: Canopy Citadel → Tuscan Infinite
Number of observations for this route: 17


,timestamp,time_slot,day_of_week,ride_name,road_distance_km,travel_time_minutes,current_price
18676,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Auto,3.154,8.75,4.08
18690,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Black,3.154,8.75,374.18
18689,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber Pet,3.154,8.75,430.00
18688,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Premier XL,3.154,8.75,247.06
18687,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Black Rentals,3.154,8.75,850.00
18686,2026-09-18 07:26:42.117985+05:30,00:00,Friday,XL Rentals,3.154,8.75,645.00
18685,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Premier Rentals,3.154,8.75,475.00
18691,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Auto Priority,3.154,8.75,109.90
18684,2026-09-18 07:26:42.117985+05:30,00:00,Friday,XL Plus Rentals,3.154,8.75,1208.65
18682,2026-09-18 07:26:42.117985+05:30,00:00,Friday,UberXL,3.154,8.75,216.72


In [12]:
# Inspect the raw Uber fare records corresponding to the suspicious ₹4.08 fare

RAW_FARES_PATH = "../data/raw/fares/uber_fares.csv"

raw_fares = pd.read_csv(RAW_FARES_PATH)

# Parse timestamps
raw_fares["timestamp"] = pd.to_datetime(raw_fares["timestamp"], errors="coerce")

# Information from the suspicious processed observation
suspicious_row = df[df["current_price"] == df["current_price"].min()].iloc[0]

target_timestamp = suspicious_row["timestamp"]
target_pickup = suspicious_row["pickup"]
target_dropoff = suspicious_row["dropoff"]
target_ride = suspicious_row["ride_name"]

print("=" * 90)
print("SUSPICIOUS PROCESSED OBSERVATION")
print("=" * 90)

print("Timestamp :", target_timestamp)
print("Pickup    :", target_pickup)
print("Dropoff   :", target_dropoff)
print("Ride      :", target_ride)
print("Fare      :", suspicious_row["current_price"])

# -------------------------------------------------------------------
# 1. Find the exact raw record
# -------------------------------------------------------------------

exact_match = raw_fares[
    (raw_fares["pickup"] == target_pickup) &
    (raw_fares["dropoff"] == target_dropoff) &
    (raw_fares["ride_name"] == target_ride) &
    (raw_fares["timestamp"] == target_timestamp)
]

print("\n" + "=" * 90)
print("EXACT RAW MATCH")
print("=" * 90)

if len(exact_match) > 0:
    display(exact_match)
else:
    print("No exact timestamp match found.")

# -------------------------------------------------------------------
# 2. Find the same route + ride within ±5 minutes
# -------------------------------------------------------------------

time_difference = (
    raw_fares["timestamp"] - target_timestamp
).abs()

nearby_same_ride = raw_fares[
    (raw_fares["pickup"] == target_pickup) &
    (raw_fares["dropoff"] == target_dropoff) &
    (raw_fares["ride_name"] == target_ride) &
    (time_difference <= pd.Timedelta(minutes=5))
].copy()

print("\n" + "=" * 90)
print("SAME ROUTE + SAME RIDE WITHIN ±5 MINUTES")
print("=" * 90)

if len(nearby_same_ride) > 0:
    display(
        nearby_same_ride.sort_values("timestamp")
    )
else:
    print("No nearby raw records found.")

# -------------------------------------------------------------------
# 3. Show every ride for this route at the suspicious timestamp
# -------------------------------------------------------------------

same_timestamp_route = raw_fares[
    (raw_fares["pickup"] == target_pickup) &
    (raw_fares["dropoff"] == target_dropoff) &
    (raw_fares["timestamp"] == target_timestamp)
].copy()

print("\n" + "=" * 90)
print("ALL RAW RIDES FOR THIS ROUTE AT THE SAME TIMESTAMP")
print("=" * 90)

if len(same_timestamp_route) > 0:
    display(
        same_timestamp_route.sort_values("current_price")
    )
else:
    print("No exact timestamp route records found.")

# -------------------------------------------------------------------
# 4. Show only the fields most useful for determining legitimacy
# -------------------------------------------------------------------

useful_columns = [
    "timestamp",
    "time_slot",
    "day_of_week",
    "provider",
    "pickup",
    "dropoff",
    "ride_name",
    "seats",
    "current_price",
    "original_price",
    "eta_minutes",
    "arrival_time",
    "promotion",
    "description"
]

available_columns = [
    col for col in useful_columns
    if col in raw_fares.columns
]

print("\n" + "=" * 90)
print("USEFUL RAW FARE DETAILS")
print("=" * 90)

display(
    same_timestamp_route[available_columns]
    .sort_values("current_price")
)

SUSPICIOUS PROCESSED OBSERVATION
Timestamp : 2026-09-18 07:26:42.117985+05:30
Pickup    : Canopy Citadel
Dropoff   : Tuscan Infinite
Ride      : Auto
Fare      : 4.08

EXACT RAW MATCH


,timestamp,time_slot,day_of_week,provider,pickup,dropoff,ride_name,seats,current_price,original_price,eta_minutes,arrival_time,promotion,description
18676,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Auto,3.0,4.08,69.08,2.0,7:49 am,Faster,NaN



SAME ROUTE + SAME RIDE WITHIN ±5 MINUTES


,timestamp,time_slot,day_of_week,provider,pickup,dropoff,ride_name,seats,current_price,original_price,eta_minutes,arrival_time,promotion,description
18676,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Auto,3.0,4.08,69.08,2.0,7:49 am,Faster,NaN



ALL RAW RIDES FOR THIS ROUTE AT THE SAME TIMESTAMP


,timestamp,time_slot,day_of_week,provider,pickup,dropoff,ride_name,seats,current_price,original_price,eta_minutes,arrival_time,promotion,description
18676,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Auto,3.0,4.08,69.08,2.0,7:49 am,Faster,NaN
18679,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Bike Saver,1.0,28.96,58.96,7.0,7:52 am,NaN,Save more when not in rush
18692,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Bike,1.0,32.21,62.21,6.0,7:52 am,NaN,Affordable 2 wheeler rides
18677,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Go Non AC,4.0,54.26,144.26,5.0,7:51 am,NaN,Everyday affordable rides
18678,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Uber Go AC,4.0,70.24,160.24,4.0,7:51 am,NaN,Affordable compact AC rides
18691,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Auto Priority,3.0,109.90,NaN,2.0,7:49 am,NaN,"Priority Matching, Faster Pick-up"
18681,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Premier AC,4.0,186.54,NaN,5.0,7:51 am,NaN,"Comfortable sedans, top-quality drivers"
18680,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Go Priority,4.0,196.80,NaN,3.0,7:50 am,NaN,Priority pickup
18682,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,UberXL,6.0,216.72,NaN,6.0,7:54 am,NaN,Affordable rides for groups up to 6
18688,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Premier XL,6.0,247.06,NaN,6.0,7:53 am,NaN,Newer XL cars with top-rated drivers



USEFUL RAW FARE DETAILS


,timestamp,time_slot,day_of_week,provider,pickup,dropoff,ride_name,seats,current_price,original_price,eta_minutes,arrival_time,promotion,description
18676,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Auto,3.0,4.08,69.08,2.0,7:49 am,Faster,NaN
18679,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Bike Saver,1.0,28.96,58.96,7.0,7:52 am,NaN,Save more when not in rush
18692,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Bike,1.0,32.21,62.21,6.0,7:52 am,NaN,Affordable 2 wheeler rides
18677,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Go Non AC,4.0,54.26,144.26,5.0,7:51 am,NaN,Everyday affordable rides
18678,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Uber Go AC,4.0,70.24,160.24,4.0,7:51 am,NaN,Affordable compact AC rides
18691,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Auto Priority,3.0,109.90,NaN,2.0,7:49 am,NaN,"Priority Matching, Faster Pick-up"
18681,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Premier AC,4.0,186.54,NaN,5.0,7:51 am,NaN,"Comfortable sedans, top-quality drivers"
18680,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Go Priority,4.0,196.80,NaN,3.0,7:50 am,NaN,Priority pickup
18682,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,UberXL,6.0,216.72,NaN,6.0,7:54 am,NaN,Affordable rides for groups up to 6
18688,2026-09-18 07:26:42.117985+05:30,00:00,Friday,Uber,Canopy Citadel,Tuscan Infinite,Premier XL,6.0,247.06,NaN,6.0,7:53 am,NaN,Newer XL cars with top-rated drivers


## 2. Final Feature Selection

Based on the EDA, the initial set of features selected for modeling is:

- `road_distance_km`
- `travel_time_minutes`
- `ride_name`
- `pickup_category`
- `drop_category`
- `time_slot`
- `day_of_week`
- `pickup`
- `dropoff`

The target variable is `current_price`.

The following features are excluded:

- `seats` — substantial systematic missingness
- `eta_minutes` — may not be available at prediction time
- `hour` — redundant with `time_slot`
- `is_weekend` — redundant with `day_of_week`
- `route` — will be evaluated separately later because of its high cardinality and overlap with pickup/dropoff

In [5]:
final_features = [
    "road_distance_km",
    "travel_time_minutes",
    "ride_name",
    "pickup_category",
    "drop_category",
    "time_slot",
    "day_of_week",
    "pickup",
    "dropoff"
]

target = "current_price"

In [6]:
print("Features:")
for feature in final_features:
    print("-", feature)

print("\nTarget:", target)

Features:
- road_distance_km
- travel_time_minutes
- ride_name
- pickup_category
- drop_category
- time_slot
- day_of_week
- pickup
- dropoff

Target: current_price


In [7]:
df[final_features + [target]].info()

<class 'pandas.DataFrame'>
Index: 27419 entries, 0 to 27419
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   road_distance_km     27419 non-null  float64
 1   travel_time_minutes  27419 non-null  float64
 2   ride_name            27419 non-null  str    
 3   pickup_category      27419 non-null  str    
 4   drop_category        27419 non-null  str    
 5   time_slot            27419 non-null  str    
 6   day_of_week          27419 non-null  str    
 7   pickup               27419 non-null  str    
 8   dropoff              27419 non-null  str    
 9   current_price        27419 non-null  float64
dtypes: float64(3), str(7)
memory usage: 2.3 MB


In [8]:
df[final_features + [target]].isnull().sum()

road_distance_km       0
travel_time_minutes    0
ride_name              0
pickup_category        0
drop_category          0
time_slot              0
day_of_week            0
pickup                 0
dropoff                0
current_price          0
dtype: int64

## 3. Define Features and Target

`X` contains the input variables used by the model.

`y` contains the target variable that the model must predict: `current_price`.

In [9]:
X = df[final_features]
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (27419, 9)
y shape: (27419,)


In [10]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

print("Minimum timestamp:", df["timestamp"].min())
print("Maximum timestamp:", df["timestamp"].max())

date_counts = (
    df["timestamp"]
    .dt.date
    .value_counts()
    .sort_index()
)

date_counts

Minimum timestamp: 2026-08-30 16:22:06.459051+05:30
Maximum timestamp: 2026-09-20 06:06:55.065200+05:30


timestamp
2026-08-30      10
2026-09-02      30
2026-09-12     158
2026-09-13    1045
2026-09-14    2468
2026-09-15    3380
2026-09-16    3318
2026-09-17    6364
2026-09-18    4659
2026-09-19    3315
2026-09-20    2672
Name: count, dtype: int64

In [13]:
# Inspect the chronological distribution of the dataset

df["timestamp"] = pd.to_datetime(df["timestamp"])

print("=" * 80)
print("DATASET TIME RANGE")
print("=" * 80)

print("Earliest observation:", df["timestamp"].min())
print("Latest observation  :", df["timestamp"].max())

print("\n" + "=" * 80)
print("OBSERVATIONS PER DATE")
print("=" * 80)

date_distribution = (
    df["timestamp"]
    .dt.date
    .value_counts()
    .sort_index()
)

display(date_distribution.to_frame(name="observations"))

print("\n" + "=" * 80)
print("CUMULATIVE DISTRIBUTION")
print("=" * 80)

cumulative_distribution = date_distribution.cumsum()

distribution_table = pd.DataFrame({
    "observations": date_distribution,
    "cumulative_observations": cumulative_distribution,
    "cumulative_percentage": (
        cumulative_distribution / len(df) * 100
    ).round(2)
})

display(distribution_table)

print("\n" + "=" * 80)
print("TOTAL OBSERVATIONS")
print("=" * 80)

print("Total:", len(df))

DATASET TIME RANGE
Earliest observation: 2026-08-30 16:22:06.459051+05:30
Latest observation  : 2026-09-20 06:06:55.065200+05:30

OBSERVATIONS PER DATE


,observations
timestamp,
2026-08-30,10
2026-09-02,30
2026-09-12,158
2026-09-13,1045
2026-09-14,2468
2026-09-15,3380
2026-09-16,3318
2026-09-17,6364
2026-09-18,4659



CUMULATIVE DISTRIBUTION


,observations,cumulative_observations,cumulative_percentage
timestamp,,,
2026-08-30,10,10,0.04
2026-09-02,30,40,0.15
2026-09-12,158,198,0.72
2026-09-13,1045,1243,4.53
2026-09-14,2468,3711,13.53
2026-09-15,3380,7091,25.86
2026-09-16,3318,10409,37.96
2026-09-17,6364,16773,61.17
2026-09-18,4659,21432,78.16



TOTAL OBSERVATIONS
Total: 27419


## 4. Prepare the Modeling Dataset

The dataset contains one anomalous zero-fare observation, which was investigated during EDA and found to be inconsistent with a normal observed fare.

The ₹4.08 observation is retained because the raw Uber record confirms that it was a genuine discounted fare.

For the modeling split, observations are ordered chronologically so that future observations are never used to train models evaluated on earlier observations.

In [18]:
# Verify the final modeling dataset

print("Zero-fare observations:", (df["current_price"] == 0).sum())
print("Dataset shape:", df.shape)
print("Minimum fare:", df["current_price"].min())
print("Maximum fare:", df["current_price"].max())

# Sort chronologically
df = df.sort_values("timestamp").reset_index(drop=True)

print("\nEarliest timestamp:", df["timestamp"].min())
print("Latest timestamp:", df["timestamp"].max())

Zero-fare observations: 0
Dataset shape: (27419, 13)
Minimum fare: 4.08
Maximum fare: 3987.61

Earliest timestamp: 2026-08-30 16:22:06.459051+05:30
Latest timestamp: 2026-09-20 06:06:55.065200+05:30


## 5. Chronological Train/Test Split

The final test set consists of observations from September 19–20, 2026.

The training set contains all observations collected up to September 18, 2026.

This simulates the real-world scenario in which the model is trained using historical observations and then asked to predict fares at a later point in time.

The final test set will remain untouched during model selection and cross-validation.

In [15]:
# Final test period starts on September 19, 2026
TEST_START_DATE = pd.Timestamp("2026-09-19", tz="Asia/Kolkata")

# Chronological split
train_df = df[df["timestamp"] < TEST_START_DATE].copy()
test_df = df[df["timestamp"] >= TEST_START_DATE].copy()

print("Training set:")
print("Shape:", train_df.shape)
print("Start:", train_df["timestamp"].min())
print("End:", train_df["timestamp"].max())

print("\nTest set:")
print("Shape:", test_df.shape)
print("Start:", test_df["timestamp"].min())
print("End:", test_df["timestamp"].max())

print("\nPercentage of data:")
print("Training:", round(len(train_df) / len(df) * 100, 2), "%")
print("Test:", round(len(test_df) / len(df) * 100, 2), "%")

Training set:
Shape: (21432, 13)
Start: 2026-08-30 16:22:06.459051+05:30
End: 2026-09-18 12:15:50.109471+05:30

Test set:
Shape: (5987, 13)
Start: 2026-09-19 16:17:44.974067+05:30
End: 2026-09-20 06:06:55.065200+05:30

Percentage of data:
Training: 78.16 %
Test: 21.84 %


In [16]:
print("Latest training timestamp:")
print(train_df["timestamp"].max())

print("\nEarliest test timestamp:")
print(test_df["timestamp"].min())

print("\nChronological ordering valid:",
      train_df["timestamp"].max() < test_df["timestamp"].min())

Latest training timestamp:
2026-09-18 12:15:50.109471+05:30

Earliest test timestamp:
2026-09-19 16:17:44.974067+05:30

Chronological ordering valid: True


## 6. Define Training and Test Features/Target

The final test set is separated before cross-validation.

Cross-validation will be performed only on the training data. Therefore, the test set remains completely unseen during model selection.

The cross-validation folds will be generated using `TimeSeriesSplit` so that each validation fold occurs after its corresponding training portion.

In [17]:
X_train = train_df[final_features]
y_train = train_df[target]

X_test = test_df[final_features]
y_test = test_df[target]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nX_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (21432, 9)
y_train: (21432,)

X_test: (5987, 9)
y_test: (5987,)


## 7. Preprocessing Pipeline

The dataset contains both numerical and categorical features.

- Numerical features are passed through unchanged.
- Categorical features are converted to one-hot encoded variables.
- `handle_unknown="ignore"` allows the model to handle categories that were not present during training.
- The preprocessing will later be combined with the regression model inside an sklearn `Pipeline` so that preprocessing is fitted separately within each cross-validation fold.

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [21]:
numerical_features = [
    "road_distance_km",
    "travel_time_minutes"
]

categorical_features = [
    "ride_name",
    "pickup_category",
    "drop_category",
    "time_slot",
    "day_of_week",
    "pickup",
    "dropoff"
]

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['road_distance_km', 'travel_time_minutes']

Categorical features:
['ride_name', 'pickup_category', 'drop_category', 'time_slot', 'day_of_week', 'pickup', 'dropoff']


In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

## 8. Baseline Model — Linear Regression

Linear Regression is used as the baseline model.

The purpose of the baseline is not necessarily to obtain the best possible predictions. Instead, it provides a simple reference point against which more complex models can be compared.

The model will be combined with the preprocessing pipeline so that categorical variables are encoded before training.

Cross-validation will be performed only on the training data using `TimeSeriesSplit`. The final test set will remain untouched until the final evaluation.

In [23]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit, cross_validate

In [24]:
baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

In [25]:
tscv = TimeSeriesSplit(n_splits=5)

In [26]:
print(baseline_pipeline)
print("\nNumber of CV splits:", tscv.n_splits)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['road_distance_km',
                                                   'travel_time_minutes']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['ride_name',
                                                   'pickup_category',
                                                   'drop_category', 'time_slot',
                                                   'day_of_week', 'pickup',
                                                   'dropoff'])])),
                ('model', LinearRegression())])

Number of CV splits: 5


## 9. Baseline Cross-Validation

The Linear Regression baseline is evaluated using 5-fold `TimeSeriesSplit`.

Cross-validation is performed only on the training data.

The final test set remains completely untouched.

We evaluate the model using:

- MAE — Mean Absolute Error
- RMSE — Root Mean Squared Error
- R² — Coefficient of Determination

In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [28]:
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

In [29]:
cv_results = cross_validate(
    baseline_pipeline,
    X_train,
    y_train,
    cv=tscv,
    scoring=scoring,
    return_train_score=False
)

In [30]:
cv_mae = -cv_results["test_MAE"]
cv_rmse = -cv_results["test_RMSE"]
cv_r2 = cv_results["test_R2"]

print("Linear Regression — Time-Series Cross-Validation")
print("=" * 55)

print("\nMAE:")
print("Fold scores:", cv_mae)
print("Mean:", cv_mae.mean())
print("Std :", cv_mae.std())

print("\nRMSE:")
print("Fold scores:", cv_rmse)
print("Mean:", cv_rmse.mean())
print("Std :", cv_rmse.std())

print("\nR²:")
print("Fold scores:", cv_r2)
print("Mean:", cv_r2.mean())
print("Std :", cv_r2.std())

Linear Regression — Time-Series Cross-Validation

MAE:
Fold scores: [172.23438164 450.02727139 219.66485465 154.61513458 141.88089417]
Mean: 227.68450728351377
Std : 114.25999235259579

RMSE:
Fold scores: [247.44841123 622.4847364  290.45957583 212.81515951 192.50139726]
Mean: 313.141856044752
Std : 158.20410771167624

R²:
Fold scores: [ 0.70554214 -0.87176878  0.52327764  0.76232159  0.87016971]
Mean: 0.39790846170642297
Std : 0.6447113797445713


### Inspecting Time-Series Cross-Validation Folds

The baseline cross-validation results show substantial variation between folds, particularly a very poor second fold.

Before comparing more complex models, we inspect the time period represented by each validation fold and compare the fare distributions.

This helps determine whether the variation is caused by temporal changes in the data rather than simply model complexity.

In [32]:
# Inspect the time period and target distribution of each CV fold

print("TIME-SERIES CV FOLD ANALYSIS")
print("=" * 80)

for fold_number, (train_idx, val_idx) in enumerate(
    tscv.split(X_train), start=1
):
    train_fold = train_df.iloc[train_idx]
    val_fold = train_df.iloc[val_idx]

    print(f"\nFOLD {fold_number}")
    print("-" * 80)

    print("Training period:")
    print("  Start:", train_fold["timestamp"].min())
    print("  End  :", train_fold["timestamp"].max())
    print("  Rows :", len(train_fold))

    print("\nValidation period:")
    print("  Start:", val_fold["timestamp"].min())
    print("  End  :", val_fold["timestamp"].max())
    print("  Rows :", len(val_fold))

    print("\nValidation fare statistics:")
    print("  Mean  :", round(val_fold["current_price"].mean(), 2))
    print("  Median:", round(val_fold["current_price"].median(), 2))
    print("  Std   :", round(val_fold["current_price"].std(), 2))
    print("  Min   :", round(val_fold["current_price"].min(), 2))
    print("  Max   :", round(val_fold["current_price"].max(), 2))

TIME-SERIES CV FOLD ANALYSIS

FOLD 1
--------------------------------------------------------------------------------
Training period:
  Start: 2026-08-30 16:22:06.459051+05:30
  End  : 2026-09-14 22:08:44.187432+05:30
  Rows : 3572

Validation period:
  Start: 2026-09-14 22:08:44.187432+05:30
  End  : 2026-09-16 00:34:32.846183+05:30
  Rows : 3572

Validation fare statistics:
  Mean  : 623.18
  Median: 506.34
  Std   : 456.07
  Min   : 14.66
  Max   : 3611.8

FOLD 2
--------------------------------------------------------------------------------
Training period:
  Start: 2026-08-30 16:22:06.459051+05:30
  End  : 2026-09-16 00:34:32.846183+05:30
  Rows : 7144

Validation period:
  Start: 2026-09-16 00:34:32.846183+05:30
  End  : 2026-09-17 04:47:38.995600+05:30
  Rows : 3572

Validation fare statistics:
  Mean  : 621.28
  Median: 499.46
  Std   : 455.05
  Min   : 10.98
  Max   : 3483.0

FOLD 3
--------------------------------------------------------------------------------
Training per

In [33]:
print("TRAINING DATA — OBSERVATIONS PER DATE")
print("=" * 60)

train_date_counts = (
    train_df
    .assign(date=train_df["timestamp"].dt.date)
    .groupby("date")
    .size()
)

print(train_date_counts)

print("\nNumber of unique dates:", len(train_date_counts))

TRAINING DATA — OBSERVATIONS PER DATE
date
2026-08-30      10
2026-09-02      30
2026-09-12     158
2026-09-13    1045
2026-09-14    2468
2026-09-15    3380
2026-09-16    3318
2026-09-17    6364
2026-09-18    4659
dtype: int64

Number of unique dates: 9


### Date-Based Expanding-Window Cross-Validation

Because the dataset was collected at an uneven rate across calendar dates, row-based `TimeSeriesSplit` can divide individual collection periods and can produce validation windows representing very different amounts of real time.

Therefore, cross-validation is performed using complete calendar dates.

The last four dates in the training period are used as validation periods. Each validation period is strictly later than its corresponding training period.

In [34]:
# Create date-based expanding-window CV folds

train_dates = train_df["timestamp"].dt.date

validation_dates = sorted(train_dates.unique())[-4:]

print("Validation dates:")
for i, date in enumerate(validation_dates, start=1):
    print(f"Fold {i}: {date}")

Validation dates:
Fold 1: 2026-09-15
Fold 2: 2026-09-16
Fold 3: 2026-09-17
Fold 4: 2026-09-18


In [35]:
print("DATE-BASED CV FOLD STRUCTURE")
print("=" * 70)

for fold_number, validation_date in enumerate(validation_dates, start=1):

    train_mask = train_dates < validation_date
    validation_mask = train_dates == validation_date

    train_fold = train_df.loc[train_mask]
    validation_fold = train_df.loc[validation_mask]

    print(f"\nFOLD {fold_number}")
    print("-" * 70)

    print("Training:")
    print("  Start:", train_fold["timestamp"].min())
    print("  End  :", train_fold["timestamp"].max())
    print("  Rows :", len(train_fold))

    print("\nValidation:")
    print("  Date :", validation_date)
    print("  Rows :", len(validation_fold))

    print("\nValidation fare statistics:")
    print("  Mean  :", round(validation_fold["current_price"].mean(), 2))
    print("  Median:", round(validation_fold["current_price"].median(), 2))
    print("  Std   :", round(validation_fold["current_price"].std(), 2))
    print("  Min   :", round(validation_fold["current_price"].min(), 2))
    print("  Max   :", round(validation_fold["current_price"].max(), 2))

DATE-BASED CV FOLD STRUCTURE

FOLD 1
----------------------------------------------------------------------
Training:
  Start: 2026-08-30 16:22:06.459051+05:30
  End  : 2026-09-14 22:08:44.187432+05:30
  Rows : 3711

Validation:
  Date : 2026-09-15
  Rows : 3380

Validation fare statistics:
  Mean  : 621.33
  Median: 505.21
  Std   : 454.76
  Min   : 14.66
  Max   : 3611.8

FOLD 2
----------------------------------------------------------------------
Training:
  Start: 2026-08-30 16:22:06.459051+05:30
  End  : 2026-09-15 23:10:31.517872+05:30
  Rows : 7091

Validation:
  Date : 2026-09-16
  Rows : 3318

Validation fare statistics:
  Mean  : 622.33
  Median: 501.58
  Std   : 455.37
  Min   : 10.98
  Max   : 3483.0

FOLD 3
----------------------------------------------------------------------
Training:
  Start: 2026-08-30 16:22:06.459051+05:30
  End  : 2026-09-16 21:13:15.890024+05:30
  Rows : 10409

Validation:
  Date : 2026-09-17
  Rows : 6364

Validation fare statistics:
  Mean  : 612

## 10. Date-Based Cross-Validation — Linear Regression

The baseline model is now evaluated using the date-based expanding-window folds defined above.

For each fold:

- All observations before the validation date are used for training.
- All observations on the validation date are used for validation.
- The preprocessing pipeline is fitted only on the corresponding training portion.
- The validation portion is used only for evaluation.

The final September 19–20 test set remains completely untouched.

In [37]:
from sklearn.base import clone

date_cv_results = []

for fold_number, validation_date in enumerate(validation_dates, start=1):

    # Create training and validation masks
    train_mask = train_df["timestamp"].dt.date < validation_date
    validation_mask = train_df["timestamp"].dt.date == validation_date

    # Split the training dataframe for this CV fold
    fold_train = train_df.loc[train_mask]
    fold_validation = train_df.loc[validation_mask]

    # Define features and target
    X_fold_train = fold_train[final_features]
    y_fold_train = fold_train[target]

    X_fold_validation = fold_validation[final_features]
    y_fold_validation = fold_validation[target]

    # Create a fresh copy of the entire pipeline
    fold_pipeline = clone(baseline_pipeline)

    # Train only on data before the validation date
    fold_pipeline.fit(X_fold_train, y_fold_train)

    # Predict the validation date
    y_pred = fold_pipeline.predict(X_fold_validation)

    # Calculate metrics
    mae = mean_absolute_error(y_fold_validation, y_pred)
    rmse = np.sqrt(
        mean_squared_error(
            y_fold_validation,
            y_pred
        )
    )
    r2 = r2_score(y_fold_validation, y_pred)

    date_cv_results.append({
        "fold": fold_number,
        "validation_date": validation_date,
        "training_rows": len(fold_train),
        "validation_rows": len(fold_validation),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

date_cv_results = pd.DataFrame(date_cv_results)

date_cv_results

,fold,validation_date,training_rows,validation_rows,MAE,RMSE,R2
0,1,2026-09-15,3711,3380,171.372913,247.633304,0.703395
1,2,2026-09-16,7091,3318,453.146479,603.464240,-0.756693
2,3,2026-09-17,10409,6364,217.126575,289.404290,0.550170
3,4,2026-09-18,16773,4659,147.990546,197.503106,0.852387


In [38]:
print("Linear Regression — Date-Based Cross-Validation")
print("=" * 65)

print("\nMean CV metrics:")
print("MAE :", round(date_cv_results["MAE"].mean(), 2))
print("RMSE:", round(date_cv_results["RMSE"].mean(), 2))
print("R²  :", round(date_cv_results["R2"].mean(), 4))

print("\nStandard deviation:")
print("MAE :", round(date_cv_results["MAE"].std(), 2))
print("RMSE:", round(date_cv_results["RMSE"].std(), 2))
print("R²  :", round(date_cv_results["R2"].std(), 4))

Linear Regression — Date-Based Cross-Validation

Mean CV metrics:
MAE : 247.41
RMSE: 334.5
R²  : 0.3373

Standard deviation:
MAE : 140.13
RMSE: 183.2
R²  : 0.7397


## 11. Stronger Model — Random Forest Regression

Random Forest Regression is evaluated as the first stronger model.

Unlike Linear Regression, Random Forest can model nonlinear relationships and interactions between features without requiring us to explicitly define those relationships.

The same preprocessing pipeline and date-based expanding-window cross-validation strategy are used so that the Random Forest results can be compared fairly with the Linear Regression baseline.

In [39]:
from sklearn.ensemble import RandomForestRegressor

In [40]:
random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [41]:
rf_cv_results = []

for fold_number, validation_date in enumerate(validation_dates, start=1):

    # Create training and validation masks
    train_mask = train_df["timestamp"].dt.date < validation_date
    validation_mask = train_df["timestamp"].dt.date == validation_date

    # Split the data
    fold_train = train_df.loc[train_mask]
    fold_validation = train_df.loc[validation_mask]

    # Features and target
    X_fold_train = fold_train[final_features]
    y_fold_train = fold_train[target]

    X_fold_validation = fold_validation[final_features]
    y_fold_validation = fold_validation[target]

    # Fresh pipeline for this fold
    fold_pipeline = clone(random_forest_pipeline)

    # Train
    fold_pipeline.fit(X_fold_train, y_fold_train)

    # Predict
    y_pred = fold_pipeline.predict(X_fold_validation)

    # Metrics
    mae = mean_absolute_error(
        y_fold_validation,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_fold_validation,
            y_pred
        )
    )

    r2 = r2_score(
        y_fold_validation,
        y_pred
    )

    rf_cv_results.append({
        "fold": fold_number,
        "validation_date": validation_date,
        "training_rows": len(fold_train),
        "validation_rows": len(fold_validation),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

rf_cv_results = pd.DataFrame(rf_cv_results)

rf_cv_results

,fold,validation_date,training_rows,validation_rows,MAE,RMSE,R2
0,1,2026-09-15,3711,3380,115.572299,208.814343,0.789098
1,2,2026-09-16,7091,3318,114.429528,201.140707,0.804839
2,3,2026-09-17,10409,6364,93.835479,171.321083,0.842362
3,4,2026-09-18,16773,4659,85.731822,144.469730,0.921018


In [42]:
print("Random Forest — Date-Based Cross-Validation")
print("=" * 65)

print("\nMean CV metrics:")
print("MAE :", round(rf_cv_results["MAE"].mean(), 2))
print("RMSE:", round(rf_cv_results["RMSE"].mean(), 2))
print("R²  :", round(rf_cv_results["R2"].mean(), 4))

print("\nStandard deviation:")
print("MAE :", round(rf_cv_results["MAE"].std(), 2))
print("RMSE:", round(rf_cv_results["RMSE"].std(), 2))
print("R²  :", round(rf_cv_results["R2"].std(), 4))

Random Forest — Date-Based Cross-Validation

Mean CV metrics:
MAE : 102.39
RMSE: 181.44
R²  : 0.8393

Standard deviation:
MAE : 14.94
RMSE: 29.48
R²  : 0.0589


## 12. Stronger Model — Gradient Boosting Regression

Gradient Boosting is evaluated as the second stronger model.

Unlike Random Forest, which builds trees independently and averages their predictions, Gradient Boosting builds trees sequentially. Each new tree attempts to correct errors made by the previous trees.

The same features, preprocessing pipeline, date-based cross-validation folds, and regression metrics are used so that the model can be compared fairly with Linear Regression and Random Forest.

In [43]:
from sklearn.ensemble import GradientBoostingRegressor

In [44]:
gradient_boosting_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            )
        )
    ]
)

In [45]:
gb_cv_results = []

for fold_number, validation_date in enumerate(validation_dates, start=1):

    # Create training and validation masks
    train_mask = train_df["timestamp"].dt.date < validation_date
    validation_mask = train_df["timestamp"].dt.date == validation_date

    # Split the data
    fold_train = train_df.loc[train_mask]
    fold_validation = train_df.loc[validation_mask]

    # Features and target
    X_fold_train = fold_train[final_features]
    y_fold_train = fold_train[target]

    X_fold_validation = fold_validation[final_features]
    y_fold_validation = fold_validation[target]

    # Fresh pipeline for this fold
    fold_pipeline = clone(gradient_boosting_pipeline)

    # Train
    fold_pipeline.fit(X_fold_train, y_fold_train)

    # Predict
    y_pred = fold_pipeline.predict(X_fold_validation)

    # Metrics
    mae = mean_absolute_error(
        y_fold_validation,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_fold_validation,
            y_pred
        )
    )

    r2 = r2_score(
        y_fold_validation,
        y_pred
    )

    gb_cv_results.append({
        "fold": fold_number,
        "validation_date": validation_date,
        "training_rows": len(fold_train),
        "validation_rows": len(fold_validation),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

gb_cv_results = pd.DataFrame(gb_cv_results)

gb_cv_results

,fold,validation_date,training_rows,validation_rows,MAE,RMSE,R2
0,1,2026-09-15,3711,3380,125.245553,216.044482,0.774240
1,2,2026-09-16,7091,3318,122.657479,193.183223,0.819975
2,3,2026-09-17,10409,6364,107.429374,171.943046,0.841216
3,4,2026-09-18,16773,4659,110.632451,161.036239,0.901865


In [46]:
print("Gradient Boosting — Date-Based Cross-Validation")
print("=" * 65)

print("\nMean CV metrics:")
print("MAE :", round(gb_cv_results["MAE"].mean(), 2))
print("RMSE:", round(gb_cv_results["RMSE"].mean(), 2))
print("R²  :", round(gb_cv_results["R2"].mean(), 4))

print("\nStandard deviation:")
print("MAE :", round(gb_cv_results["MAE"].std(), 2))
print("RMSE:", round(gb_cv_results["RMSE"].std(), 2))
print("R²  :", round(gb_cv_results["R2"].std(), 4))

Gradient Boosting — Date-Based Cross-Validation

Mean CV metrics:
MAE : 116.49
RMSE: 185.55
R²  : 0.8343

Standard deviation:
MAE : 8.78
RMSE: 24.32
R²  : 0.053


## 13. Random Forest Hyperparameter Tuning

The initial Random Forest performed substantially better than Linear Regression and slightly better than the initial Gradient Boosting model.

We now tune the Random Forest hyperparameters using the same date-based cross-validation strategy.

The final test period (September 19–20) remains completely untouched during tuning.

In [47]:
from sklearn.model_selection import RandomizedSearchCV

In [48]:
# Prepare the training data used for hyperparameter tuning

X_tuning = train_df[final_features]
y_tuning = train_df[target]

# Create date-based CV splits
date_cv_splits = []

for validation_date in validation_dates:

    train_indices = train_df.index[
        train_df["timestamp"].dt.date < validation_date
    ]

    validation_indices = train_df.index[
        train_df["timestamp"].dt.date == validation_date
    ]

    # Convert dataframe indices to positional indices
    train_positions = X_tuning.index.get_indexer(train_indices)
    validation_positions = X_tuning.index.get_indexer(validation_indices)

    date_cv_splits.append(
        (
            train_positions,
            validation_positions
        )
    )

print("Number of date-based CV folds:", len(date_cv_splits))

for i, (train_idx, val_idx) in enumerate(date_cv_splits, start=1):
    print(
        f"Fold {i}: "
        f"training rows = {len(train_idx)}, "
        f"validation rows = {len(val_idx)}"
    )

Number of date-based CV folds: 4
Fold 1: training rows = 3711, validation rows = 3380
Fold 2: training rows = 7091, validation rows = 3318
Fold 3: training rows = 10409, validation rows = 6364
Fold 4: training rows = 16773, validation rows = 4659


In [49]:
rf_parameter_distributions = {
    "model__n_estimators": [200, 300, 400],
    "model__max_depth": [None, 15, 25],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [1.0, "sqrt"]
}

In [50]:
rf_random_search = RandomizedSearchCV(
    estimator=random_forest_pipeline,
    param_distributions=rf_parameter_distributions,
    n_iter=12,
    scoring="neg_mean_absolute_error",
    cv=date_cv_splits,
    random_state=42,
    n_jobs=-1,
    refit=False
)

In [51]:
print("Starting Random Forest hyperparameter search...")
print("Number of configurations:", 12)
print("Number of CV folds:", len(date_cv_splits))
print("Total model fits:", 12 * len(date_cv_splits))

rf_random_search.fit(X_tuning, y_tuning)

print("\nRandom Forest hyperparameter search completed.")

Starting Random Forest hyperparameter search...
Number of configurations: 12
Number of CV folds: 4
Total model fits: 48

Random Forest hyperparameter search completed.


In [52]:
print("BEST RANDOM FOREST PARAMETERS")
print("=" * 60)

print(rf_random_search.best_params_)

print("\nBest CV MAE:")
print(round(-rf_random_search.best_score_, 2))

BEST RANDOM FOREST PARAMETERS
{'model__n_estimators': 400, 'model__min_samples_leaf': 2, 'model__max_features': 1.0, 'model__max_depth': None}

Best CV MAE:
104.45


In [53]:
rf_tuning_results = pd.DataFrame(
    rf_random_search.cv_results_
)

rf_tuning_results = rf_tuning_results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "param_model__max_features"
    ]
].copy()

rf_tuning_results["mean_MAE"] = (
    -rf_tuning_results["mean_test_score"]
)

rf_tuning_results["std_MAE"] = (
    rf_tuning_results["std_test_score"]
)

rf_tuning_results = rf_tuning_results.sort_values(
    "rank_test_score"
)

rf_tuning_results

,rank_test_score,mean_test_score,std_test_score,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,mean_MAE,std_MAE
5,1,-104.446819,10.952460,400,None,2,1.0,104.446819,10.952460
8,2,-104.623443,10.822236,200,None,2,1.0,104.623443,10.822236
11,3,-106.305117,9.467425,400,None,4,1.0,106.305117,9.467425
4,4,-106.341464,9.476790,400,25,4,1.0,106.341464,9.476790
0,5,-109.829453,8.413151,300,15,1,1.0,109.829453,8.413151
10,6,-128.359928,11.160654,300,None,2,sqrt,128.359928,11.160654
3,7,-129.126749,11.306346,200,None,2,sqrt,129.126749,11.306346
6,8,-139.562241,11.464821,400,None,4,sqrt,139.562241,11.464821
1,9,-152.838626,8.828205,300,25,2,sqrt,152.838626,8.828205
2,10,-153.191983,8.372594,200,25,2,sqrt,153.191983,8.372594


## 13.1 Targeted Random Forest Tuning

The initial randomized search did not outperform the original Random Forest configuration.

Therefore, a focused search is performed around the promising region identified by the first search.

The search includes the original baseline configuration so that the tuned model can be compared directly against it.

In [54]:
rf_targeted_parameters = {
    "model__n_estimators": [200, 400, 600],
    "model__max_depth": [None, 25],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": [1.0]
}

In [55]:
from sklearn.model_selection import GridSearchCV

rf_grid_search = GridSearchCV(
    estimator=random_forest_pipeline,
    param_grid=rf_targeted_parameters,
    scoring="neg_mean_absolute_error",
    cv=date_cv_splits,
    n_jobs=-1,
    refit=False
)

In [56]:
print("Starting targeted Random Forest search...")
print("Number of configurations:", 12)
print("Number of CV folds:", len(date_cv_splits))
print("Total model fits:", 12 * len(date_cv_splits))

rf_grid_search.fit(X_tuning, y_tuning)

print("\nTargeted Random Forest search completed.")

Starting targeted Random Forest search...
Number of configurations: 12
Number of CV folds: 4
Total model fits: 48


/Users/mayanksharma/Desktop/Bangalore_Cab_Fare_Prediction/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Targeted Random Forest search completed.


In [57]:
print("BEST TARGETED RANDOM FOREST PARAMETERS")
print("=" * 65)

print(rf_grid_search.best_params_)

print("\nBest CV MAE:")
print(round(-rf_grid_search.best_score_, 2))

BEST TARGETED RANDOM FOREST PARAMETERS
{'model__max_depth': None, 'model__max_features': 1.0, 'model__min_samples_leaf': 1, 'model__n_estimators': 600}

Best CV MAE:
102.19


In [58]:
rf_targeted_results = pd.DataFrame(
    rf_grid_search.cv_results_
)

rf_targeted_results = rf_targeted_results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "param_model__max_features"
    ]
].copy()

rf_targeted_results["mean_MAE"] = (
    -rf_targeted_results["mean_test_score"]
)

rf_targeted_results["std_MAE"] = (
    rf_targeted_results["std_test_score"]
)

rf_targeted_results = rf_targeted_results.sort_values(
    "rank_test_score"
)

rf_targeted_results

,rank_test_score,mean_test_score,std_test_score,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,mean_MAE,std_MAE
2,1,-102.191519,13.054996,600,None,1,1.0,102.191519,13.054996
1,2,-102.285117,12.969515,400,None,1,1.0,102.285117,12.969515
0,3,-102.392282,12.936363,200,None,1,1.0,102.392282,12.936363
8,4,-102.923750,12.610991,600,25,1,1.0,102.923750,12.610991
7,5,-103.028747,12.543752,400,25,1,1.0,103.028747,12.543752
6,6,-103.103303,12.578773,200,25,1,1.0,103.103303,12.578773
5,7,-104.418357,11.007800,600,None,2,1.0,104.418357,11.007800
4,8,-104.446819,10.952460,400,None,2,1.0,104.446819,10.952460
3,9,-104.623443,10.822236,200,None,2,1.0,104.623443,10.822236
11,10,-104.709470,10.963486,600,25,2,1.0,104.709470,10.963486


## 13.2 Evaluate the Tuned Random Forest

The targeted search identified a Random Forest with 600 trees, unrestricted tree depth, and a minimum leaf size of 1 as the best configuration according to mean CV MAE.

The selected configuration is now evaluated using MAE, RMSE, and R² across the same date-based validation folds used for all previous models.

In [59]:
tuned_random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=600,
                max_depth=None,
                min_samples_leaf=1,
                max_features=1.0,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [60]:
tuned_rf_cv_results = []

for fold_number, validation_date in enumerate(validation_dates, start=1):

    # Create training and validation masks
    train_mask = train_df["timestamp"].dt.date < validation_date
    validation_mask = train_df["timestamp"].dt.date == validation_date

    # Split data
    fold_train = train_df.loc[train_mask]
    fold_validation = train_df.loc[validation_mask]

    # Features and target
    X_fold_train = fold_train[final_features]
    y_fold_train = fold_train[target]

    X_fold_validation = fold_validation[final_features]
    y_fold_validation = fold_validation[target]

    # Fresh pipeline for this fold
    fold_pipeline = clone(tuned_random_forest_pipeline)

    # Train
    fold_pipeline.fit(
        X_fold_train,
        y_fold_train
    )

    # Predict
    y_pred = fold_pipeline.predict(
        X_fold_validation
    )

    # Metrics
    mae = mean_absolute_error(
        y_fold_validation,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_fold_validation,
            y_pred
        )
    )

    r2 = r2_score(
        y_fold_validation,
        y_pred
    )

    tuned_rf_cv_results.append({
        "fold": fold_number,
        "validation_date": validation_date,
        "training_rows": len(fold_train),
        "validation_rows": len(fold_validation),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

tuned_rf_cv_results = pd.DataFrame(
    tuned_rf_cv_results
)

tuned_rf_cv_results

,fold,validation_date,training_rows,validation_rows,MAE,RMSE,R2
0,1,2026-09-15,3711,3380,115.553795,209.229185,0.788259
1,2,2026-09-16,7091,3318,114.402668,201.197046,0.804730
2,3,2026-09-17,10409,6364,93.083736,169.753644,0.845234
3,4,2026-09-18,16773,4659,85.725876,143.948062,0.921587


In [61]:
print("Tuned Random Forest — Date-Based Cross-Validation")
print("=" * 65)

print("\nMean CV metrics:")
print("MAE :", round(tuned_rf_cv_results["MAE"].mean(), 2))
print("RMSE:", round(tuned_rf_cv_results["RMSE"].mean(), 2))
print("R²  :", round(tuned_rf_cv_results["R2"].mean(), 4))

print("\nStandard deviation:")
print("MAE :", round(tuned_rf_cv_results["MAE"].std(), 2))
print("RMSE:", round(tuned_rf_cv_results["RMSE"].std(), 2))
print("R²  :", round(tuned_rf_cv_results["R2"].std(), 4))

Tuned Random Forest — Date-Based Cross-Validation

Mean CV metrics:
MAE : 102.19
RMSE: 181.03
R²  : 0.84

Standard deviation:
MAE : 15.07
RMSE: 30.02
R²  : 0.0595


## Gradient Boosting — Hyperparameter Tuning

The baseline Gradient Boosting model performed reasonably well, but its performance was still slightly behind the Random Forest model.

The baseline results were:

- **MAE:** ₹116.49
- **RMSE:** ₹185.55
- **R²:** 0.8343

To determine whether Gradient Boosting can be improved, we perform a targeted hyperparameter search.

The hyperparameters being tuned are:

- **`n_estimators`** — number of boosting stages.
- **`learning_rate`** — contribution of each boosting stage to the final prediction.
- **`max_depth`** — maximum depth of the individual decision trees.
- **`min_samples_leaf`** — minimum number of samples required in a leaf.

We use the same **date-based cross-validation strategy** used for the other models. This ensures that the model is always trained on earlier observations and validated on later observations.

The search uses **16 hyperparameter configurations across 4 date-based CV folds**, resulting in **64 model fits**.

The primary metric for selecting the best configuration is **MAE**, since it directly represents the average absolute prediction error in rupees.

In [62]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

In [63]:
gb_param_grid = {
    "model__n_estimators": [200, 400],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3],
    "model__min_samples_leaf": [1, 2]
}

total_configurations = (
    len(gb_param_grid["model__n_estimators"])
    * len(gb_param_grid["model__learning_rate"])
    * len(gb_param_grid["model__max_depth"])
    * len(gb_param_grid["model__min_samples_leaf"])
)

print("Number of Gradient Boosting configurations:", total_configurations)

Number of Gradient Boosting configurations: 16


In [64]:
gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingRegressor(
                random_state=42
            )
        )
    ]
)

print(gb_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['road_distance_km',
                                                   'travel_time_minutes']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['ride_name',
                                                   'pickup_category',
                                                   'drop_category', 'time_slot',
                                                   'day_of_week', 'pickup',
                                                   'dropoff'])])),
                ('model', GradientBoostingRegressor(random_state=42))])


In [65]:
print("Starting Gradient Boosting hyperparameter search...")
print("Number of configurations:", total_configurations)
print("Number of CV folds:", len(date_cv_splits))
print("Total model fits:", total_configurations * len(date_cv_splits))

gb_grid_search = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    scoring="neg_mean_absolute_error",
    cv=date_cv_splits,
    n_jobs=-1,
    return_train_score=False
)

gb_grid_search.fit(X_train, y_train)

print("\nGradient Boosting hyperparameter search completed.")

Starting Gradient Boosting hyperparameter search...
Number of configurations: 16
Number of CV folds: 4
Total model fits: 64

Gradient Boosting hyperparameter search completed.


In [66]:
print("BEST GRADIENT BOOSTING PARAMETERS")
print("=" * 70)

print(gb_grid_search.best_params_)

print("\nBest CV MAE:")
print(round(-gb_grid_search.best_score_, 2))

BEST GRADIENT BOOSTING PARAMETERS
{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__min_samples_leaf': 1, 'model__n_estimators': 400}

Best CV MAE:
110.47


In [67]:
gb_cv_results = pd.DataFrame(gb_grid_search.cv_results_)

gb_results = gb_cv_results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_model__n_estimators",
        "param_model__learning_rate",
        "param_model__max_depth",
        "param_model__min_samples_leaf"
    ]
].copy()

gb_results["mean_MAE"] = -gb_results["mean_test_score"]
gb_results["std_MAE"] = gb_results["std_test_score"]

gb_results = gb_results.sort_values("rank_test_score")

gb_results

,rank_test_score,mean_test_score,std_test_score,param_model__n_estimators,param_model__learning_rate,param_model__max_depth,param_model__min_samples_leaf,mean_MAE,std_MAE
5,1,-110.468367,10.368073,400,0.05,3,1,110.468367,10.368073
7,2,-110.552189,10.239078,400,0.05,3,2,110.552189,10.239078
14,3,-110.679490,10.375970,200,0.10,3,2,110.679490,10.375970
12,4,-111.132055,10.793343,200,0.10,3,1,111.132055,10.793343
11,5,-114.272851,10.603188,400,0.10,2,2,114.272851,10.603188
15,6,-114.504446,12.814277,400,0.10,3,2,114.504446,12.814277
9,7,-115.002956,11.758050,400,0.10,2,1,115.002956,11.758050
10,8,-115.074326,8.642298,200,0.10,2,2,115.074326,8.642298
3,9,-115.266947,8.495900,400,0.05,2,2,115.266947,8.495900
1,10,-115.291760,8.574018,400,0.05,2,1,115.291760,8.574018


## Model Comparison

The models developed so far are compared using the same date-based cross-validation strategy.

The comparison uses three regression metrics:

- **MAE (Mean Absolute Error):** average absolute prediction error in rupees. Lower is better.
- **RMSE (Root Mean Squared Error):** gives greater importance to large prediction errors. Lower is better.
- **R²:** proportion of target variance explained by the model. Higher is better.

Both the mean and standard deviation across CV folds are reported. The mean represents average performance, while the standard deviation indicates how much performance varies across different validation periods.

The models being compared are:

1. Linear Regression
2. Random Forest
3. Tuned Random Forest
4. Gradient Boosting
5. Tuned Gradient Boosting

The CV results are used for **model comparison and model selection**. Since hyperparameter tuning was also performed using these date-based CV folds, these CV metrics should not be treated as the final unbiased estimate of model performance.

The final test set remains completely untouched during model development and selection. It will be used only once, after the final model has been selected, to obtain the final performance estimate.

## 14. Evaluate the Tuned Gradient Boosting Model

The hyperparameter search identified the following Gradient Boosting configuration:

- `n_estimators = 400`
- `learning_rate = 0.05`
- `max_depth = 3`
- `min_samples_leaf = 1`

The best configuration achieved a cross-validation MAE of approximately ₹110.47 during hyperparameter selection.

The selected configuration is now evaluated using MAE, RMSE, and R² across the same four date-based validation folds used for the other models.

This provides a directly comparable evaluation of the tuned Gradient Boosting model.

In [68]:
tuned_gradient_boosting_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingRegressor(
                n_estimators=400,
                learning_rate=0.05,
                max_depth=3,
                min_samples_leaf=1,
                random_state=42
            )
        )
    ]
)

print(tuned_gradient_boosting_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['road_distance_km',
                                                   'travel_time_minutes']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['ride_name',
                                                   'pickup_category',
                                                   'drop_category', 'time_slot',
                                                   'day_of_week', 'pickup',
                                                   'dropoff'])])),
                ('model',
                 GradientBoostingRegressor(learning_rate=0.05, n_estimators=400,
                                           random_state=42))])


In [69]:
tuned_gb_cv_results = []

for fold_number, validation_date in enumerate(validation_dates, start=1):

    # Create training and validation masks
    train_mask = train_df["timestamp"].dt.date < validation_date
    validation_mask = train_df["timestamp"].dt.date == validation_date

    # Split data
    fold_train = train_df.loc[train_mask]
    fold_validation = train_df.loc[validation_mask]

    # Features and target
    X_fold_train = fold_train[final_features]
    y_fold_train = fold_train[target]

    X_fold_validation = fold_validation[final_features]
    y_fold_validation = fold_validation[target]

    # Fresh pipeline for this fold
    fold_pipeline = clone(tuned_gradient_boosting_pipeline)

    # Train
    fold_pipeline.fit(
        X_fold_train,
        y_fold_train
    )

    # Predict
    y_pred = fold_pipeline.predict(
        X_fold_validation
    )

    # Metrics
    mae = mean_absolute_error(
        y_fold_validation,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_fold_validation,
            y_pred
        )
    )

    r2 = r2_score(
        y_fold_validation,
        y_pred
    )

    tuned_gb_cv_results.append({
        "fold": fold_number,
        "validation_date": validation_date,
        "training_rows": len(fold_train),
        "validation_rows": len(fold_validation),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

tuned_gb_cv_results = pd.DataFrame(
    tuned_gb_cv_results
)

tuned_gb_cv_results

,fold,validation_date,training_rows,validation_rows,MAE,RMSE,R2
0,1,2026-09-15,3711,3380,123.371032,211.118950,0.784417
1,2,2026-09-16,7091,3318,117.427829,189.298273,0.827143
2,3,2026-09-17,10409,6364,103.522505,165.256205,0.853326
3,4,2026-09-18,16773,4659,97.552102,143.756885,0.921795


In [70]:
print("Tuned Gradient Boosting — Date-Based Cross-Validation")
print("=" * 65)

print("\nMean CV metrics:")
print("MAE :", round(tuned_gb_cv_results["MAE"].mean(), 2))
print("RMSE:", round(tuned_gb_cv_results["RMSE"].mean(), 2))
print("R²  :", round(tuned_gb_cv_results["R2"].mean(), 4))

print("\nStandard deviation:")
print("MAE :", round(tuned_gb_cv_results["MAE"].std(), 2))
print("RMSE:", round(tuned_gb_cv_results["RMSE"].std(), 2))
print("R²  :", round(tuned_gb_cv_results["R2"].std(), 4))

Tuned Gradient Boosting — Date-Based Cross-Validation

Mean CV metrics:
MAE : 110.47
RMSE: 177.36
R²  : 0.8467

Standard deviation:
MAE : 11.97
RMSE: 29.2
R²  : 0.0576


### Recreating Baseline Gradient Boosting Results

The original Gradient Boosting fold-level results were stored in `gb_cv_results`, but that variable was later reused for the hyperparameter-search results.

To keep the model comparison unambiguous, the baseline Gradient Boosting model is evaluated again using the same date-based CV procedure.

In [71]:
baseline_gb_cv_results = []

for fold_number, validation_date in enumerate(validation_dates, start=1):

    train_mask = train_df["timestamp"].dt.date < validation_date
    validation_mask = train_df["timestamp"].dt.date == validation_date

    fold_train = train_df.loc[train_mask]
    fold_validation = train_df.loc[validation_mask]

    X_fold_train = fold_train[final_features]
    y_fold_train = fold_train[target]

    X_fold_validation = fold_validation[final_features]
    y_fold_validation = fold_validation[target]

    fold_pipeline = clone(gradient_boosting_pipeline)

    fold_pipeline.fit(
        X_fold_train,
        y_fold_train
    )

    y_pred = fold_pipeline.predict(
        X_fold_validation
    )

    mae = mean_absolute_error(
        y_fold_validation,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_fold_validation,
            y_pred
        )
    )

    r2 = r2_score(
        y_fold_validation,
        y_pred
    )

    baseline_gb_cv_results.append({
        "fold": fold_number,
        "validation_date": validation_date,
        "training_rows": len(fold_train),
        "validation_rows": len(fold_validation),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

baseline_gb_cv_results = pd.DataFrame(
    baseline_gb_cv_results
)

baseline_gb_cv_results

,fold,validation_date,training_rows,validation_rows,MAE,RMSE,R2
0,1,2026-09-15,3711,3380,125.245553,216.044482,0.774240
1,2,2026-09-16,7091,3318,122.657479,193.183223,0.819975
2,3,2026-09-17,10409,6364,107.429374,171.943046,0.841216
3,4,2026-09-18,16773,4659,110.632451,161.036239,0.901865


## 15. Model Comparison

All models are now compared using the same date-based cross-validation procedure.

The final comparison includes:

- Linear Regression
- Random Forest
- Tuned Random Forest
- Gradient Boosting
- Tuned Gradient Boosting

The final test set remains completely untouched and will only be used after model selection.

In [72]:
model_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Tuned Random Forest",
        "Gradient Boosting",
        "Tuned Gradient Boosting"
    ],

    "Mean MAE": [
        date_cv_results["MAE"].mean(),
        rf_cv_results["MAE"].mean(),
        tuned_rf_cv_results["MAE"].mean(),
        baseline_gb_cv_results["MAE"].mean(),
        tuned_gb_cv_results["MAE"].mean()
    ],

    "Std MAE": [
        date_cv_results["MAE"].std(),
        rf_cv_results["MAE"].std(),
        tuned_rf_cv_results["MAE"].std(),
        baseline_gb_cv_results["MAE"].std(),
        tuned_gb_cv_results["MAE"].std()
    ],

    "Mean RMSE": [
        date_cv_results["RMSE"].mean(),
        rf_cv_results["RMSE"].mean(),
        tuned_rf_cv_results["RMSE"].mean(),
        baseline_gb_cv_results["RMSE"].mean(),
        tuned_gb_cv_results["RMSE"].mean()
    ],

    "Std RMSE": [
        date_cv_results["RMSE"].std(),
        rf_cv_results["RMSE"].std(),
        tuned_rf_cv_results["RMSE"].std(),
        baseline_gb_cv_results["RMSE"].std(),
        tuned_gb_cv_results["RMSE"].std()
    ],

    "Mean R2": [
        date_cv_results["R2"].mean(),
        rf_cv_results["R2"].mean(),
        tuned_rf_cv_results["R2"].mean(),
        baseline_gb_cv_results["R2"].mean(),
        tuned_gb_cv_results["R2"].mean()
    ],

    "Std R2": [
        date_cv_results["R2"].std(),
        rf_cv_results["R2"].std(),
        tuned_rf_cv_results["R2"].std(),
        baseline_gb_cv_results["R2"].std(),
        tuned_gb_cv_results["R2"].std()
    ]
})

model_comparison.round(2)

,Model,Mean MAE,Std MAE,Mean RMSE,Std RMSE,Mean R2,Std R2
0,Linear Regression,247.41,140.13,334.50,183.20,0.34,0.74
1,Random Forest,102.39,14.94,181.44,29.48,0.84,0.06
2,Tuned Random Forest,102.19,15.07,181.03,30.02,0.84,0.06
3,Gradient Boosting,116.49,8.78,185.55,24.32,0.83,0.05
4,Tuned Gradient Boosting,110.47,11.97,177.36,29.20,0.85,0.06


In [73]:
model_comparison_sorted = (
    model_comparison
    .sort_values("Mean MAE")
    .reset_index(drop=True)
)

model_comparison_sorted.round(2)

,Model,Mean MAE,Std MAE,Mean RMSE,Std RMSE,Mean R2,Std R2
0,Tuned Random Forest,102.19,15.07,181.03,30.02,0.84,0.06
1,Random Forest,102.39,14.94,181.44,29.48,0.84,0.06
2,Tuned Gradient Boosting,110.47,11.97,177.36,29.20,0.85,0.06
3,Gradient Boosting,116.49,8.78,185.55,24.32,0.83,0.05
4,Linear Regression,247.41,140.13,334.50,183.20,0.34,0.74


## Final Model Evaluation on the Test Set

The Tuned Random Forest was selected during date-based cross-validation using MAE as the primary model-selection metric.

The selected hyperparameters are:

- `n_estimators = 600`
- `max_depth = None`
- `min_samples_leaf = 1`
- `max_features = 1.0`

The final model is now trained on the complete development/training dataset.

The final test set has remained completely untouched throughout model development, cross-validation, and hyperparameter tuning.

Therefore, the test-set metrics provide the final estimate of how the selected model performs on unseen future observations.

In [74]:
# Create the final tuned Random Forest pipeline

final_rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=600,
                max_depth=None,
                min_samples_leaf=1,
                max_features=1.0,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Final Tuned Random Forest pipeline created.")

Final Tuned Random Forest pipeline created.


In [75]:
# Train the final model on all development/training data

final_rf_pipeline.fit(X_train, y_train)

print("Final Tuned Random Forest trained successfully.")

Final Tuned Random Forest trained successfully.


In [76]:
# Predict on the untouched final test set

y_test_pred = final_rf_pipeline.predict(X_test)

# Calculate final test metrics

test_mae = mean_absolute_error(y_test, y_test_pred)

test_rmse = np.sqrt(
    mean_squared_error(y_test, y_test_pred)
)

test_r2 = r2_score(y_test, y_test_pred)

print("=" * 70)
print("FINAL TUNED RANDOM FOREST — TEST SET PERFORMANCE")
print("=" * 70)

print(f"\nMAE  : ₹{test_mae:.2f}")
print(f"RMSE : ₹{test_rmse:.2f}")
print(f"R²   : {test_r2:.4f}")

FINAL TUNED RANDOM FOREST — TEST SET PERFORMANCE

MAE  : ₹68.99
RMSE : ₹130.62
R²   : 0.9249


## Final Test Evaluation

The Tuned Random Forest was selected using date-based cross-validation, with MAE used as the primary model-selection metric.

The selected model was then trained on the complete development dataset and evaluated once on the previously untouched final test set.

The final test performance was:

- MAE: ₹68.99
- RMSE: ₹130.62
- R²: 0.9249

These results represent the final performance estimate of the selected model on unseen future observations.

The final test set will not be used for further model development or feature-selection decisions.

## Error Analysis

The final test-set performance has been frozen and will not be used for further model development.

Error analysis is therefore performed on the development data using the selected Tuned Random Forest model.

The purpose of this analysis is to understand:

- where the model makes larger prediction errors,
- which ride types are more difficult to predict,
- whether prediction errors vary with trip distance,
- whether certain time slots are harder to predict,
- whether the model systematically overpredicts or underpredicts fares,
- and whether particular groups of observations contribute disproportionately to the prediction error.

These findings will be used to identify possible areas for model improvement.

In [77]:
# Generate predictions on the development dataset

y_train_pred = final_rf_pipeline.predict(X_train)

# Calculate prediction errors

development_errors = y_train - y_train_pred
development_absolute_errors = np.abs(development_errors)

print("Development predictions generated successfully.")
print(f"Number of observations: {len(y_train_pred)}")

Development predictions generated successfully.
Number of observations: 21432


In [78]:
# Create a DataFrame for error analysis

error_analysis_df = X_train.copy()

error_analysis_df["actual_fare"] = y_train.values
error_analysis_df["predicted_fare"] = y_train_pred
error_analysis_df["error"] = development_errors.values
error_analysis_df["absolute_error"] = development_absolute_errors.values

error_analysis_df.head()

,road_distance_km,travel_time_minutes,ride_name,pickup_category,drop_category,time_slot,day_of_week,pickup,dropoff,actual_fare,predicted_fare,error,absolute_error
0,23.757,50.42,Go Non AC,mall,residential_area,15:00,Sunday,Brand Factory,NCS Elite,528.89,478.70550,50.18450,50.18450
1,23.757,50.42,Uber Pet,mall,residential_area,15:00,Sunday,Brand Factory,NCS Elite,1143.39,1082.40295,60.98705,60.98705
2,23.757,50.42,Premier XL,mall,residential_area,15:00,Sunday,Brand Factory,NCS Elite,1188.17,1115.74325,72.42675,72.42675
3,23.757,50.42,Auto Priority,mall,residential_area,15:00,Sunday,Brand Factory,NCS Elite,712.28,659.70190,52.57810,52.57810
4,23.757,50.42,UberXL,mall,residential_area,15:00,Sunday,Brand Factory,NCS Elite,1046.93,970.49800,76.43200,76.43200


In [79]:
# Observations with the largest prediction errors

largest_errors = (
    error_analysis_df
    .sort_values("absolute_error", ascending=False)
    .head(20)
)

largest_errors[
    [
        "ride_name",
        "pickup",
        "dropoff",
        "road_distance_km",
        "travel_time_minutes",
        "time_slot",
        "day_of_week",
        "actual_fare",
        "predicted_fare",
        "error",
        "absolute_error"
    ]
]

,ride_name,pickup,dropoff,road_distance_km,travel_time_minutes,time_slot,day_of_week,actual_fare,predicted_fare,error,absolute_error
6080,Auto Priority,Light and Matter Physics,Delta Electronics Bommasandra,28.033,57.66,18:00,Tuesday,3091.10,2219.862567,871.237433,871.237433
6081,Bike,Light and Matter Physics,Delta Electronics Bommasandra,28.033,57.66,18:00,Tuesday,2015.48,1342.702100,672.777900,672.777900
6072,Auto,Light and Matter Physics,Delta Electronics Bommasandra,28.033,57.66,18:00,Tuesday,2095.70,1457.868950,637.831050,637.831050
1168,XL Rentals,Sudha Sadan Apartments,Huskur Road,25.030,51.63,21:00,Sunday,3509.30,2909.236267,600.063733,600.063733
6090,Bike Saver,Light and Matter Physics,Delta Electronics Bommasandra,28.033,57.66,18:00,Tuesday,1975.53,1377.452233,598.077767,598.077767
6054,XL Plus Rentals,Nexus Whitefield,Madavara,35.464,79.45,18:00,Tuesday,3523.85,2992.173000,531.677000,531.677000
5258,Black Rentals,Wipro Ahmed Plaza,Sudha Sadan Apartments,22.123,51.01,15:00,Tuesday,2736.35,2211.579333,524.770667,524.770667
6052,XL Rentals,Nexus Whitefield,Madavara,35.464,79.45,18:00,Tuesday,3611.80,3102.587100,509.212900,509.212900
15124,XL Rentals,Organic Chemistry,Hosa Road,21.907,50.24,18:00,Thursday,2850.95,2357.159533,493.790467,493.790467
1179,Go Rentals,Sudha Sadan Apartments,Huskur Road,25.030,51.63,21:00,Sunday,2562.00,2071.883333,490.116667,490.116667


## Error Analysis by Ride Type

The first investigation examines whether prediction errors differ across ride types.

This is important because different ride types can have substantially different pricing structures. In particular, premium and rental services may behave differently from standard point-to-point rides.

For each ride type, we examine:

- number of observations,
- average fare,
- mean absolute error (MAE),
- RMSE,
- and average signed error.

The signed error helps identify systematic underprediction or overprediction.

In [80]:
# Error analysis by ride type

ride_error_analysis = (
    error_analysis_df
    .groupby("ride_name")
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=("error", lambda x: np.sqrt(np.mean(x ** 2))),
        mean_error=("error", "mean")
    )
    .sort_values("MAE", ascending=False)
)

ride_error_analysis.round(2)

,observations,mean_actual_fare,MAE,RMSE,mean_error
ride_name,,,,,
XL Rentals,1310,1246.83,54.93,90.79,8.83
Black Rentals,334,1146.65,52.35,91.36,15.11
Premier Rentals,1310,990.58,48.93,79.75,7.62
Black,904,694.37,45.13,65.17,7.35
Go Rentals,1310,828.13,42.55,71.23,7.58
Comfort,6,1283.61,41.12,52.77,-40.31
XL,109,2059.87,38.27,58.67,3.30
Uber Go AC,1311,331.80,30.90,46.82,4.60
Innova,48,2468.55,29.54,41.95,3.73


In [81]:
# Ride types with the highest average absolute error

ride_error_analysis.sort_values(
    "MAE",
    ascending=False
).head(10).round(2)

,observations,mean_actual_fare,MAE,RMSE,mean_error
ride_name,,,,,
XL Rentals,1310,1246.83,54.93,90.79,8.83
Black Rentals,334,1146.65,52.35,91.36,15.11
Premier Rentals,1310,990.58,48.93,79.75,7.62
Black,904,694.37,45.13,65.17,7.35
Go Rentals,1310,828.13,42.55,71.23,7.58
Comfort,6,1283.61,41.12,52.77,-40.31
XL,109,2059.87,38.27,58.67,3.30
Uber Go AC,1311,331.80,30.90,46.82,4.60
Innova,48,2468.55,29.54,41.95,3.73


## Error Analysis by Time Slot

The next analysis examines whether prediction accuracy changes across the predefined collection time slots.

This can reveal whether the model struggles more during particular periods of the day, potentially because fare patterns vary more strongly during those periods.

In [82]:
# Error analysis by time slot

time_error_analysis = (
    error_analysis_df
    .groupby("time_slot")
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=("error", lambda x: np.sqrt(np.mean(x ** 2))),
        mean_error=("error", "mean")
    )
    .sort_values("MAE", ascending=False)
)

time_error_analysis.round(2)

,observations,mean_actual_fare,MAE,RMSE,mean_error
time_slot,,,,,
18:00,2641,671.83,36.74,71.44,12.63
09:00,4156,706.42,33.10,59.76,8.93
12:00,1307,791.08,26.95,47.59,-6.04
00:00,6327,581.52,26.65,44.89,0.33
15:00,3273,592.21,24.59,47.41,1.63
21:00,3728,589.91,23.00,44.20,2.82


## Error Analysis by Trip Distance

Prediction errors are also examined across different trip-distance ranges.

This helps determine whether the model performs differently for short, medium, and long trips.

Distance ranges are created only for analysis; the original continuous distance feature remains unchanged in the model.

In [83]:
# Create distance ranges for error analysis

error_analysis_df["distance_group"] = pd.cut(
    error_analysis_df["road_distance_km"],
    bins=[0, 5, 10, 20, 30, 50, np.inf],
    labels=[
        "0-5 km",
        "5-10 km",
        "10-20 km",
        "20-30 km",
        "30-50 km",
        "50+ km"
    ]
)

distance_error_analysis = (
    error_analysis_df
    .groupby("distance_group", observed=False)
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=("error", lambda x: np.sqrt(np.mean(x ** 2))),
        mean_error=("error", "mean")
    )
)

distance_error_analysis.round(2)

,observations,mean_actual_fare,MAE,RMSE,mean_error
distance_group,,,,,
0-5 km,890,340.26,11.23,19.83,0.66
5-10 km,4519,408.39,20.59,37.23,3.22
10-20 km,9793,580.23,25.39,46.45,3.55
20-30 km,4619,815.68,41.33,73.49,5.18
30-50 km,1308,1101.13,36.74,59.78,4.58
50+ km,303,1724.40,46.37,65.90,2.62


## Prediction Bias

The signed prediction error is examined to determine whether the model systematically overpredicts or underpredicts fares.

A positive error indicates underprediction, while a negative error indicates overprediction.

In [84]:
# Overall prediction bias

mean_prediction_error = error_analysis_df["error"].mean()

print(f"Mean prediction error: ₹{mean_prediction_error:.2f}")

if mean_prediction_error > 0:
    print("The model tends to underpredict fares on average.")
elif mean_prediction_error < 0:
    print("The model tends to overpredict fares on average.")
else:
    print("The model has approximately zero average prediction bias.")

Mean prediction error: ₹3.76
The model tends to underpredict fares on average.


## Out-of-Fold Error Analysis

The previous error analysis used predictions on the training data. Since the final Random Forest was fitted on those observations, these predictions can be overly optimistic.

To obtain a more reliable estimate of model errors on unseen observations, out-of-fold (OOF) predictions are generated using the same date-based cross-validation strategy used during model evaluation.

Each observation receives a prediction from a model that was not trained on that observation.

These OOF predictions will be used for detailed error analysis by:

- ride type,
- time slot,
- trip distance,
- and prediction bias.

In [87]:
# Generate out-of-fold predictions using the same date-based CV folds

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd


# Final tuned Random Forest parameters
final_rf_params = {
    "n_estimators": 600,
    "max_depth": None,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "random_state": 42,
}


# Store OOF predictions
oof_predictions = []

print("Generating out-of-fold predictions...")
print(f"Number of CV folds: {len(date_cv_splits)}")

for fold_number, (train_idx, validation_idx) in enumerate(
    date_cv_splits,
    start=1
):

    # Create a fresh copy of the final tuned pipeline
    fold_model = clone(
        final_rf_pipeline
    )

    # Replace the model with the final tuned Random Forest
    fold_model.set_params(
        model__n_estimators=final_rf_params["n_estimators"],
        model__max_depth=final_rf_params["max_depth"],
        model__min_samples_leaf=final_rf_params["min_samples_leaf"],
        model__max_features=final_rf_params["max_features"],
    )

    # Split the development data according to this fold
    X_train_fold = X_tuning.iloc[train_idx]
    y_train_fold = y_tuning.iloc[train_idx]

    X_validation_fold = X_tuning.iloc[validation_idx]
    y_validation_fold = y_tuning.iloc[validation_idx]

    # Train only on the training portion of this fold
    fold_model.fit(
        X_train_fold,
        y_train_fold
    )

    # Predict the observations that were NOT used for training
    validation_predictions = fold_model.predict(
        X_validation_fold
    )

    # Store actual and predicted values
    fold_results = X_validation_fold.copy()

    fold_results["actual_fare"] = y_validation_fold.values
    fold_results["predicted_fare"] = validation_predictions
    fold_results["error"] = (
        fold_results["actual_fare"]
        - fold_results["predicted_fare"]
    )
    fold_results["absolute_error"] = (
        fold_results["error"].abs()
    )
    fold_results["fold"] = fold_number

    oof_predictions.append(
        fold_results
    )

    print(
        f"Fold {fold_number}: "
        f"{len(validation_idx)} OOF predictions generated."
    )


# Combine predictions from all folds
oof_predictions_df = pd.concat(
    oof_predictions,
    ignore_index=True
)

print("\nOut-of-fold predictions generated successfully.")
print(
    f"Number of OOF observations: "
    f"{len(oof_predictions_df)}"
)

Generating out-of-fold predictions...
Number of CV folds: 4
Fold 1: 3380 OOF predictions generated.
Fold 2: 3318 OOF predictions generated.
Fold 3: 6364 OOF predictions generated.
Fold 4: 4659 OOF predictions generated.

Out-of-fold predictions generated successfully.
Number of OOF observations: 17721


## Out-of-Fold Error Analysis

Out-of-fold (OOF) predictions are used to analyze model errors on the development data without evaluating observations on models that were trained on those same observations.

The analysis will examine:

- Overall OOF prediction performance
- Prediction bias
- Error distribution
- Error by ride type
- Error by time slot
- Error by trip distance

The final test set remains untouched during this analysis.

In [88]:
# Overall out-of-fold performance

oof_mae = mean_absolute_error(
    oof_predictions_df["actual_fare"],
    oof_predictions_df["predicted_fare"]
)

oof_rmse = np.sqrt(
    mean_squared_error(
        oof_predictions_df["actual_fare"],
        oof_predictions_df["predicted_fare"]
    )
)

oof_r2 = r2_score(
    oof_predictions_df["actual_fare"],
    oof_predictions_df["predicted_fare"]
)


print("OVERALL OUT-OF-FOLD PERFORMANCE")
print("=" * 60)

print(f"MAE  : ₹{oof_mae:.2f}")
print(f"RMSE : ₹{oof_rmse:.2f}")
print(f"R²   : {oof_r2:.4f}")

OVERALL OUT-OF-FOLD PERFORMANCE
MAE  : ₹99.43
RMSE : ₹178.12
R²   : 0.8529


## OOF Prediction Bias

Prediction bias measures whether the model systematically tends to predict fares above or below the actual fares.

The prediction error is defined as:

**error = actual fare − predicted fare**

Therefore:

- Positive mean error → the model tends to underpredict fares.
- Negative mean error → the model tends to overpredict fares.
- A value close to zero → little overall systematic bias.

This analysis uses only out-of-fold predictions from the development data.

In [89]:
# Overall OOF prediction bias

oof_mean_error = oof_predictions_df["error"].mean()

print("OOF PREDICTION BIAS")
print("=" * 60)

print(f"Mean prediction error: ₹{oof_mean_error:.2f}")

if oof_mean_error > 0:
    print("The model tends to underpredict fares on average.")
elif oof_mean_error < 0:
    print("The model tends to overpredict fares on average.")
else:
    print("The model has approximately zero average prediction bias.")

OOF PREDICTION BIAS
Mean prediction error: ₹15.30
The model tends to underpredict fares on average.


## OOF Error Distribution

The distribution of absolute prediction errors is examined to understand how frequently the model makes small, moderate, and large errors.

Absolute error is used because it measures the magnitude of the prediction error regardless of whether the model overpredicts or underpredicts.

The analysis reports selected percentiles and the proportion of observations within different error ranges.

In [90]:
# Distribution of OOF absolute errors

error_percentiles = (
    oof_predictions_df["absolute_error"]
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99])
)

print("OOF ABSOLUTE ERROR DISTRIBUTION")
print("=" * 60)

print(f"Median absolute error : ₹{error_percentiles.loc[0.50]:.2f}")
print(f"75th percentile      : ₹{error_percentiles.loc[0.75]:.2f}")
print(f"90th percentile      : ₹{error_percentiles.loc[0.90]:.2f}")
print(f"95th percentile      : ₹{error_percentiles.loc[0.95]:.2f}")
print(f"99th percentile      : ₹{error_percentiles.loc[0.99]:.2f}")


# Error ranges

error_ranges = pd.Series({
    "≤ ₹50": (
        oof_predictions_df["absolute_error"] <= 50
    ).mean() * 100,

    "≤ ₹100": (
        oof_predictions_df["absolute_error"] <= 100
    ).mean() * 100,

    "≤ ₹200": (
        oof_predictions_df["absolute_error"] <= 200
    ).mean() * 100,

    "≤ ₹300": (
        oof_predictions_df["absolute_error"] <= 300
    ).mean() * 100,

    "> ₹300": (
        oof_predictions_df["absolute_error"] > 300
    ).mean() * 100
})


print("\nPROPORTION OF OOF PREDICTIONS")
print("=" * 60)

for error_range, percentage in error_ranges.items():
    print(f"{error_range:<10}: {percentage:.2f}%")

OOF ABSOLUTE ERROR DISTRIBUTION
Median absolute error : ₹49.12
75th percentile      : ₹117.34
90th percentile      : ₹247.84
95th percentile      : ₹376.23
99th percentile      : ₹733.06

PROPORTION OF OOF PREDICTIONS
≤ ₹50     : 50.51%
≤ ₹100    : 70.71%
≤ ₹200    : 86.68%
≤ ₹300    : 92.46%
> ₹300    : 7.54%


## OOF Error Analysis by Ride Type

Prediction errors are analyzed separately for each ride type to identify whether certain services are systematically harder to predict.

For each ride type, the analysis reports:

- Number of observations
- Mean actual fare
- MAE
- RMSE
- Mean prediction error

Ride types with very few observations should be interpreted cautiously because their error estimates are less reliable.

In [91]:
# OOF error analysis by ride type

ride_error_analysis = (
    oof_predictions_df
    .groupby("ride_name")
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=(
            "error",
            lambda x: np.sqrt(np.mean(x ** 2))
        ),
        mean_error=("error", "mean")
    )
    .sort_values("MAE", ascending=False)
)

ride_error_analysis.round(2)

,observations,mean_actual_fare,MAE,RMSE,mean_error
ride_name,,,,,
Comfort,6,1283.61,279.91,318.57,244.26
Innova,39,2351.45,237.27,320.38,172.44
XL,100,2041.00,213.44,313.35,120.06
XL Rentals,1082,1247.55,171.90,270.60,19.95
Black Rentals,293,1147.94,161.83,284.28,52.61
Premier Rentals,1082,991.86,153.21,242.40,20.57
Black,742,693.17,142.02,206.15,31.13
Go Rentals,1082,832.96,134.81,218.88,30.34
Hatchback,100,1084.33,128.81,175.72,-4.13


## OOF Error Analysis by Time Slot

Prediction errors are analyzed across the different fare-collection time slots.

This helps determine whether model performance changes depending on the time of day.

For each time slot, the analysis reports:

- Number of observations
- Mean actual fare
- MAE
- RMSE
- Mean prediction error

A positive mean prediction error indicates systematic underprediction, while a negative value indicates systematic overprediction.

In [92]:
# OOF error analysis by time slot

time_error_analysis = (
    oof_predictions_df
    .groupby("time_slot")
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=(
            "error",
            lambda x: np.sqrt(np.mean(x ** 2))
        ),
        mean_error=("error", "mean")
    )
    .sort_values("MAE", ascending=False)
)

time_error_analysis.round(2)

,observations,mean_actual_fare,MAE,RMSE,mean_error
time_slot,,,,,
18:00,1844,689.72,150.46,275.59,59.86
09:00,3673,724.43,113.25,199.02,52.24
12:00,957,844.36,102.55,170.87,-14.28
21:00,2938,562.48,87.43,145.57,-29.76
00:00,5132,583.15,86.91,147.21,-0.05
15:00,3177,593.68,84.19,153.90,22.09


## Findings from OOF Error Analysis by Time Slot

The OOF predictions show substantial variation in model performance across time slots.

The largest errors occur at **18:00** and **09:00**, with MAE values of ₹150.46 and ₹113.25 respectively.

The model also shows systematic underprediction during these periods, with mean prediction errors of approximately ₹59.86 at 18:00 and ₹52.24 at 09:00.

In contrast, the model performs better at 15:00, 21:00 and 00:00, where MAE is below ₹90.

This indicates that the model does not capture all time-dependent fare variation equally well. In particular, the higher errors and positive prediction bias at 09:00 and 18:00 suggest that additional temporal information or feature interactions may help improve predictions during these periods.

The time-slot analysis therefore provides a potential direction for model improvement, but no feature changes are made yet. Further error analysis is performed before deciding on improvements.

## OOF Error Analysis by Trip Distance

Prediction errors are analyzed across different road-distance ranges.

This helps determine whether the model performs differently for short, medium, and long trips.

For each distance group, the analysis reports:

- Number of observations
- Mean actual fare
- MAE
- RMSE
- Mean prediction error

This analysis helps identify whether prediction errors increase with trip distance and whether the model systematically underpredicts or overpredicts fares for particular distance ranges.

In [93]:
# Create distance groups

error_analysis_df["distance_group"] = pd.cut(
    error_analysis_df["road_distance_km"],
    bins=[0, 5, 10, 20, 30, 50, np.inf],
    labels=[
        "0-5 km",
        "5-10 km",
        "10-20 km",
        "20-30 km",
        "30-50 km",
        "50+ km"
    ],
    right=False
)


# Calculate error metrics for each distance group

distance_error_analysis = (
    error_analysis_df
    .groupby("distance_group", observed=True)
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=(
            "error",
            lambda x: np.sqrt(np.mean(x ** 2))
        ),
        mean_error=("error", "mean")
    )
)


distance_error_analysis.round(2)

,observations,mean_actual_fare,MAE,RMSE,mean_error
distance_group,,,,,
0-5 km,890,340.26,11.23,19.83,0.66
5-10 km,4519,408.39,20.59,37.23,3.22
10-20 km,9793,580.23,25.39,46.45,3.55
20-30 km,4619,815.68,41.33,73.49,5.18
30-50 km,1308,1101.13,36.74,59.78,4.58
50+ km,303,1724.40,46.37,65.90,2.62


## Findings from OOF Error Analysis by Trip Distance

The OOF analysis shows that prediction error generally increases as trip distance increases.

Short trips are predicted relatively accurately, with an MAE of ₹11.23 for trips below 5 km. The MAE increases to ₹41.33 for trips between 20 and 30 km and ₹46.37 for trips above 50 km.

However, the mean prediction error remains relatively small across all distance groups. Therefore, the increase in error with distance does not appear to be caused by strong systematic overprediction or underprediction.

Instead, longer trips appear to contain greater fare variability that is more difficult for the current feature set to capture accurately.

The 50+ km group contains only 303 observations, so its error estimate should be interpreted with more caution than the larger distance groups.

## OOF Error Analysis by Route

The model is further evaluated at the route level by combining the pickup and dropoff locations into a route identifier.

This analysis helps determine whether certain pickup-to-dropoff combinations have substantially larger prediction errors.

Because some routes have very few observations, only routes with a sufficient number of OOF observations are considered when identifying high-error routes.

This prevents conclusions from being driven by routes with only one or two observations.

In [94]:
# Create an explicit route identifier

error_analysis_df["route"] = (
    error_analysis_df["pickup"]
    + " → "
    + error_analysis_df["dropoff"]
)


# Analyze route-level errors

route_error_analysis = (
    error_analysis_df
    .groupby("route")
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=(
            "error",
            lambda x: np.sqrt(np.mean(x ** 2))
        ),
        mean_error=("error", "mean")
    )
)


# Consider only routes with at least 20 observations

route_error_analysis_filtered = (
    route_error_analysis[
        route_error_analysis["observations"] >= 20
    ]
    .sort_values("MAE", ascending=False)
)


route_error_analysis_filtered.head(20).round(2)

,observations,mean_actual_fare,MAE,RMSE,mean_error
route,,,,,
Light and Matter Physics → Delta Electronics Bommasandra,30,1238.65,217.16,315.63,44.39
Department of Youth Services and Sports → Dasarahalli,30,1290.70,135.27,179.16,95.36
Department of Youth Services and Sports → Delta Electronics Bommasandra,30,974.72,125.67,153.32,34.64
Sudha Sadan Apartments → Huskur Road,75,1038.87,123.25,184.34,26.32
Nokia Care → Angsana Oasis Spa & Resort,26,900.18,97.93,139.21,34.50
Rashmi Rise → DreamQuest Infotech,30,884.84,89.13,118.35,17.58
Nandishwara Amusement Park → Kingsdale,30,1431.39,87.26,107.38,73.15
Whitefield (Kadugodi) → Vijaya Pre-University College,30,1026.32,86.84,117.69,31.55
Chickpet Cross → Whitefield (Kadugodi),31,843.14,85.86,123.18,17.71


## Model Improvement: Route Feature

The OOF error analysis showed that prediction error varies substantially across individual pickup-to-dropoff routes.

The current model represents `pickup` and `dropoff` as separate categorical features. However, this does not explicitly represent the interaction between a particular pickup location and a particular dropoff location.

A new `route` feature is therefore introduced by combining the pickup and dropoff locations into a single categorical feature.

The effect of this feature will be evaluated using the same date-based cross-validation strategy used throughout the notebook.

The route feature will only be retained if it improves out-of-sample cross-validation performance without introducing evidence of overfitting.

In [95]:
# Create a route feature by combining pickup and dropoff

X_tuning_route = X_tuning.copy()

X_tuning_route["route"] = (
    X_tuning_route["pickup"]
    + " → "
    + X_tuning_route["dropoff"]
)

print("Original number of features:", X_tuning.shape[1])
print("New number of features:", X_tuning_route.shape[1])

print("\nNumber of unique routes:")
print(X_tuning_route["route"].nunique())

print("\nSample routes:")
display(
    X_tuning_route[
        ["pickup", "dropoff", "route"]
    ].head()
)

Original number of features: 9
New number of features: 10

Number of unique routes:
1022

Sample routes:


,pickup,dropoff,route
0,Brand Factory,NCS Elite,Brand Factory → NCS Elite
1,Brand Factory,NCS Elite,Brand Factory → NCS Elite
2,Brand Factory,NCS Elite,Brand Factory → NCS Elite
3,Brand Factory,NCS Elite,Brand Factory → NCS Elite
4,Brand Factory,NCS Elite,Brand Factory → NCS Elite


In [96]:
# Preprocessing pipeline with the new route feature

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor


# Numerical features
numerical_features_route = [
    "road_distance_km",
    "travel_time_minutes"
]


# Categorical features including the new route feature
categorical_features_route = [
    "ride_name",
    "pickup_category",
    "drop_category",
    "time_slot",
    "day_of_week",
    "pickup",
    "dropoff",
    "route"
]


# Preprocessor
route_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numerical_features_route
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features_route
        )
    ]
)


# Random Forest model with the same tuned parameters
route_rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            route_preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=600,
                max_depth=None,
                min_samples_leaf=1,
                max_features=1.0,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


print("Route-aware Random Forest pipeline created successfully.")

Route-aware Random Forest pipeline created successfully.


In [99]:
# Add route feature to the development/tuning dataset

X_tuning_route = X_tuning.copy()

X_tuning_route["route"] = (
    X_tuning_route["pickup"]
    + " → "
    + X_tuning_route["dropoff"]
)

print("Number of features:", X_tuning_route.shape[1])
print("Route feature added successfully.")
print("\nRoute examples:")
display(
    X_tuning_route[
        ["pickup", "dropoff", "route"]
    ].head()
)

Number of features: 10
Route feature added successfully.

Route examples:


,pickup,dropoff,route
0,Brand Factory,NCS Elite,Brand Factory → NCS Elite
1,Brand Factory,NCS Elite,Brand Factory → NCS Elite
2,Brand Factory,NCS Elite,Brand Factory → NCS Elite
3,Brand Factory,NCS Elite,Brand Factory → NCS Elite
4,Brand Factory,NCS Elite,Brand Factory → NCS Elite


In [100]:
# Evaluate the route-aware Random Forest using the same date-based CV folds

route_cv_results = []

print("Evaluating route-aware Random Forest...")
print(f"Number of CV folds: {len(date_cv_splits)}")

for fold_number, (train_idx, validation_idx) in enumerate(
    date_cv_splits,
    start=1
):
    
    # Split development data according to the existing fold
    X_train_fold = X_tuning_route.iloc[train_idx]
    y_train_fold = y_tuning.iloc[train_idx]

    X_validation_fold = X_tuning_route.iloc[validation_idx]
    y_validation_fold = y_tuning.iloc[validation_idx]

    # Train route-aware model
    route_rf_pipeline.fit(
        X_train_fold,
        y_train_fold
    )

    # Predict validation data
    validation_predictions = route_rf_pipeline.predict(
        X_validation_fold
    )

    # Calculate metrics
    fold_mae = mean_absolute_error(
        y_validation_fold,
        validation_predictions
    )

    fold_rmse = np.sqrt(
        mean_squared_error(
            y_validation_fold,
            validation_predictions
        )
    )

    fold_r2 = r2_score(
        y_validation_fold,
        validation_predictions
    )

    route_cv_results.append({
        "fold": fold_number,
        "MAE": fold_mae,
        "RMSE": fold_rmse,
        "R2": fold_r2
    })

    print(
        f"Fold {fold_number}: "
        f"MAE = ₹{fold_mae:.2f}, "
        f"RMSE = ₹{fold_rmse:.2f}, "
        f"R² = {fold_r2:.4f}"
    )


# Convert results to DataFrame
route_cv_results_df = pd.DataFrame(route_cv_results)


print("\n" + "=" * 70)
print("ROUTE-AWARE RANDOM FOREST — DATE-BASED CROSS-VALIDATION")
print("=" * 70)

print(
    f"\nMean MAE  : ₹{route_cv_results_df['MAE'].mean():.2f}"
)

print(
    f"Mean RMSE : ₹{route_cv_results_df['RMSE'].mean():.2f}"
)

print(
    f"Mean R²   : {route_cv_results_df['R2'].mean():.4f}"
)

print("\nStandard deviation:")

print(
    f"MAE  : ₹{route_cv_results_df['MAE'].std():.2f}"
)

print(
    f"RMSE : ₹{route_cv_results_df['RMSE'].std():.2f}"
)

print(
    f"R²   : {route_cv_results_df['R2'].std():.4f}"
)

Evaluating route-aware Random Forest...
Number of CV folds: 4
Fold 1: MAE = ₹115.97, RMSE = ₹210.40, R² = 0.7859
Fold 2: MAE = ₹111.10, RMSE = ₹193.32, R² = 0.8197
Fold 3: MAE = ₹90.64, RMSE = ₹167.00, R² = 0.8502
Fold 4: MAE = ₹82.93, RMSE = ₹141.40, R² = 0.9243

ROUTE-AWARE RANDOM FOREST — DATE-BASED CROSS-VALIDATION

Mean MAE  : ₹100.16
Mean RMSE : ₹178.03
Mean R²   : 0.8450

Standard deviation:
MAE  : ₹15.89
RMSE : ₹30.25
R²   : 0.0590


In [101]:
# Add route feature to the final test set

X_test_route = X_test.copy()

X_test_route["route"] = (
    X_test_route["pickup"]
    + " → "
    + X_test_route["dropoff"]
)

print("Test features:", X_test_route.shape[1])
print("Route feature added to test set successfully.")

display(
    X_test_route[
        ["pickup", "dropoff", "route"]
    ].head()
)

Test features: 10
Route feature added to test set successfully.


,pickup,dropoff,route
21432,VRL Bus Station,Rashmi Rise,VRL Bus Station → Rashmi Rise
21433,VRL Bus Station,Rashmi Rise,VRL Bus Station → Rashmi Rise
21434,VRL Bus Station,Rashmi Rise,VRL Bus Station → Rashmi Rise
21435,VRL Bus Station,Rashmi Rise,VRL Bus Station → Rashmi Rise
21436,VRL Bus Station,Rashmi Rise,VRL Bus Station → Rashmi Rise


In [103]:
# Create the final route-aware Random Forest pipeline

from sklearn.base import clone

final_route_rf_pipeline = clone(
    route_rf_pipeline
)

# Use the final tuned Random Forest parameters
final_route_rf_pipeline.set_params(
    model__n_estimators=600,
    model__max_depth=None,
    model__min_samples_leaf=1,
    model__max_features=1.0
)

# Train on ALL development data
final_route_rf_pipeline.fit(
    X_tuning_route,
    y_tuning
)

print("Final route-aware Random Forest trained successfully.")

Final route-aware Random Forest trained successfully.


In [104]:
# Final test-set evaluation of the route-aware Random Forest

final_route_test_predictions = final_route_rf_pipeline.predict(
    X_test_route
)

final_route_test_mae = mean_absolute_error(
    y_test,
    final_route_test_predictions
)

final_route_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        final_route_test_predictions
    )
)

final_route_test_r2 = r2_score(
    y_test,
    final_route_test_predictions
)


print("=" * 70)
print("FINAL ROUTE-AWARE RANDOM FOREST — TEST SET PERFORMANCE")
print("=" * 70)

print(f"\nMAE  : ₹{final_route_test_mae:.2f}")
print(f"RMSE : ₹{final_route_test_rmse:.2f}")
print(f"R²   : {final_route_test_r2:.4f}")

FINAL ROUTE-AWARE RANDOM FOREST — TEST SET PERFORMANCE

MAE  : ₹68.46
RMSE : ₹131.00
R²   : 0.9245


## Target Transformation — Log-Transformed Fare

The fare distribution is right-skewed, with a relatively small number of high-fare observations.

To investigate whether this affects model performance, a second version of the target will be created using a logarithmic transformation:

\[
y_{\text{log}} = \log(1+y)
\]

The transformed target reduces the influence of very large fare values during training.

After prediction, the transformation will be reversed using:

\[
\hat{y} = \exp(\hat{y}_{\text{log}})-1
\]

The log-transformed model will be evaluated using the same date-based cross-validation strategy and the same untouched final test set as the original model.

The original 9-feature set will be used. The `route` feature will not be included.

In [105]:
# Create log-transformed target variables

y_tuning_log = np.log1p(y_tuning)
y_test_log = np.log1p(y_test)

print("Original target:")
print(f"  Mean   : ₹{y_tuning.mean():.2f}")
print(f"  Median : ₹{y_tuning.median():.2f}")
print(f"  Maximum: ₹{y_tuning.max():.2f}")

print("\nLog-transformed target:")
print(f"  Mean   : {y_tuning_log.mean():.4f}")
print(f"  Median : {y_tuning_log.median():.4f}")
print(f"  Maximum: {y_tuning_log.max():.4f}")

Original target:
  Mean   : ₹632.74
  Median : ₹514.54
  Maximum: ₹3611.80

Log-transformed target:
  Mean   : 6.1723
  Median : 6.2452
  Maximum: 8.1922


In [106]:
# Create Random Forest pipeline for the log-transformed target

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor


# Numerical features
numeric_features = [
    "road_distance_km",
    "travel_time_minutes"
]

# Categorical features
categorical_features = [
    "ride_name",
    "pickup_category",
    "drop_category",
    "time_slot",
    "day_of_week",
    "pickup",
    "dropoff"
]


# Preprocessing
log_rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)


# Random Forest with the same tuned parameters
log_rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            log_rf_preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=600,
                max_depth=None,
                min_samples_leaf=1,
                max_features=1.0,
                random_state=42
            )
        )
    ]
)

print("Log-target Random Forest pipeline created successfully.")

Log-target Random Forest pipeline created successfully.


In [107]:
# Evaluate log-target Random Forest using the same date-based CV folds

log_rf_cv_results = []

print("Evaluating log-target Random Forest...")
print(f"Number of CV folds: {len(date_cv_splits)}")

for fold_number, (train_idx, validation_idx) in enumerate(
    date_cv_splits,
    start=1
):

    # Split development data according to the existing fold
    X_train_fold = X_tuning.iloc[train_idx]
    X_validation_fold = X_tuning.iloc[validation_idx]

    y_train_fold_log = y_tuning_log.iloc[train_idx]
    y_validation_fold = y_tuning.iloc[validation_idx]

    # Train using the log-transformed target
    log_rf_pipeline.fit(
        X_train_fold,
        y_train_fold_log
    )

    # Predict in log space
    validation_predictions_log = log_rf_pipeline.predict(
        X_validation_fold
    )

    # Convert predictions back to rupees
    validation_predictions = np.expm1(
        validation_predictions_log
    )

    # Calculate metrics in original rupee scale
    fold_mae = mean_absolute_error(
        y_validation_fold,
        validation_predictions
    )

    fold_rmse = np.sqrt(
        mean_squared_error(
            y_validation_fold,
            validation_predictions
        )
    )

    fold_r2 = r2_score(
        y_validation_fold,
        validation_predictions
    )

    log_rf_cv_results.append({
        "fold": fold_number,
        "MAE": fold_mae,
        "RMSE": fold_rmse,
        "R2": fold_r2
    })

    print(
        f"Fold {fold_number}: "
        f"MAE = ₹{fold_mae:.2f}, "
        f"RMSE = ₹{fold_rmse:.2f}, "
        f"R² = {fold_r2:.4f}"
    )


# Convert results to DataFrame
log_rf_cv_results_df = pd.DataFrame(
    log_rf_cv_results
)


print("\n" + "=" * 70)
print("LOG-TARGET RANDOM FOREST — DATE-BASED CROSS-VALIDATION")
print("=" * 70)

print(
    f"\nMean MAE  : "
    f"₹{log_rf_cv_results_df['MAE'].mean():.2f}"
)

print(
    f"Mean RMSE : "
    f"₹{log_rf_cv_results_df['RMSE'].mean():.2f}"
)

print(
    f"Mean R²   : "
    f"{log_rf_cv_results_df['R2'].mean():.4f}"
)

print("\nStandard deviation:")

print(
    f"MAE  : "
    f"₹{log_rf_cv_results_df['MAE'].std():.2f}"
)

print(
    f"RMSE : "
    f"₹{log_rf_cv_results_df['RMSE'].std():.2f}"
)

print(
    f"R²   : "
    f"{log_rf_cv_results_df['R2'].std():.4f}"
)

Evaluating log-target Random Forest...
Number of CV folds: 4
Fold 1: MAE = ₹115.77, RMSE = ₹210.93, R² = 0.7848
Fold 2: MAE = ₹104.60, RMSE = ₹182.05, R² = 0.8401
Fold 3: MAE = ₹89.32, RMSE = ₹161.93, R² = 0.8592
Fold 4: MAE = ₹81.46, RMSE = ₹135.61, R² = 0.9304

LOG-TARGET RANDOM FOREST — DATE-BASED CROSS-VALIDATION

Mean MAE  : ₹97.79
Mean RMSE : ₹172.63
Mean R²   : 0.8536

Standard deviation:
MAE  : ₹15.37
RMSE : ₹31.84
R²   : 0.0601


## Final Log-Target Random Forest — Test Set Evaluation

The log-target Random Forest showed improved performance during date-based cross-validation.

The final model will now be trained on the complete development dataset using the selected hyperparameters.

The model will then be evaluated once on the untouched test set.

Predictions are converted back from log space to the original fare scale before calculating MAE, RMSE, and R².

In [108]:
# Train the final log-target Random Forest on all development data

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


# Final log-target Random Forest
final_log_rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=600,
                max_depth=None,
                min_samples_leaf=1,
                max_features=1.0,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


print("Training final log-target Random Forest...")

# Train using the complete development dataset
final_log_rf_pipeline.fit(
    X_tuning,
    np.log1p(y_tuning)
)

print("Final log-target Random Forest trained successfully.")


# Predict the test set in log space
log_test_predictions = final_log_rf_pipeline.predict(
    X_test
)


# Convert predictions back to the original fare scale
y_test_pred_log = np.expm1(
    log_test_predictions
)


# Calculate test-set metrics on the original fare scale
final_log_mae = mean_absolute_error(
    y_test,
    y_test_pred_log
)

final_log_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred_log
    )
)

final_log_r2 = r2_score(
    y_test,
    y_test_pred_log
)


print("\n" + "=" * 70)
print("FINAL LOG-TARGET RANDOM FOREST — TEST SET PERFORMANCE")
print("=" * 70)

print(f"\nMAE  : ₹{final_log_mae:.2f}")
print(f"RMSE : ₹{final_log_rmse:.2f}")
print(f"R²   : {final_log_r2:.4f}")

Training final log-target Random Forest...
Final log-target Random Forest trained successfully.

FINAL LOG-TARGET RANDOM FOREST — TEST SET PERFORMANCE

MAE  : ₹67.38
RMSE : ₹129.48
R²   : 0.9262


## Final Model — Error Analysis

The log-target Random Forest achieved the strongest overall performance on the untouched test set.

The next step is to analyze where the model performs well and where prediction errors remain.

The analysis will examine:

- overall error distribution
- prediction bias
- error by ride type
- error by time slot
- error by trip distance
- largest individual prediction errors

The purpose is to understand the model's weaknesses rather than simply optimize the overall metrics.

In [109]:
# Create error-analysis dataframe for the final log-target Random Forest

error_analysis_df = X_test.copy()

error_analysis_df["actual_fare"] = y_test.values
error_analysis_df["predicted_fare"] = y_test_pred_log

# Prediction error
# Positive error  -> model underpredicted
# Negative error  -> model overpredicted
error_analysis_df["error"] = (
    error_analysis_df["actual_fare"]
    - error_analysis_df["predicted_fare"]
)

# Absolute error
error_analysis_df["absolute_error"] = (
    error_analysis_df["error"].abs()
)

print("Final error-analysis dataframe created successfully.")
print(f"Number of test observations: {len(error_analysis_df)}")

print("\nColumns:")
print(error_analysis_df.columns.tolist())

print("\nFirst 5 observations:")
display(
    error_analysis_df.head()
)

Final error-analysis dataframe created successfully.
Number of test observations: 5987

Columns:
['road_distance_km', 'travel_time_minutes', 'ride_name', 'pickup_category', 'drop_category', 'time_slot', 'day_of_week', 'pickup', 'dropoff', 'actual_fare', 'predicted_fare', 'error', 'absolute_error']

First 5 observations:


,road_distance_km,travel_time_minutes,ride_name,pickup_category,drop_category,time_slot,day_of_week,pickup,dropoff,actual_fare,predicted_fare,error,absolute_error
21432,11.807,27.74,UberXL,bus_terminal,residential_area,15:00,Saturday,VRL Bus Station,Rashmi Rise,490.12,487.703881,2.416119,2.416119
21433,11.807,27.74,Uber Go AC,bus_terminal,residential_area,15:00,Saturday,VRL Bus Station,Rashmi Rise,229.87,252.276628,-22.406628,22.406628
21434,11.807,27.74,Go Non AC,bus_terminal,residential_area,15:00,Saturday,VRL Bus Station,Rashmi Rise,200.05,216.465589,-16.415589,16.415589
21435,11.807,27.74,Bike Saver,bus_terminal,residential_area,15:00,Saturday,VRL Bus Station,Rashmi Rise,136.01,145.011519,-9.001519,9.001519
21436,11.807,27.74,Auto,bus_terminal,residential_area,15:00,Saturday,VRL Bus Station,Rashmi Rise,184.93,189.751626,-4.821626,4.821626


In [110]:
# Overall error distribution and prediction bias

mean_prediction_error = error_analysis_df["error"].mean()
median_absolute_error = error_analysis_df["absolute_error"].median()

percentile_75 = error_analysis_df["absolute_error"].quantile(0.75)
percentile_90 = error_analysis_df["absolute_error"].quantile(0.90)
percentile_95 = error_analysis_df["absolute_error"].quantile(0.95)
percentile_99 = error_analysis_df["absolute_error"].quantile(0.99)


print("=" * 70)
print("FINAL MODEL — OVERALL ERROR ANALYSIS")
print("=" * 70)

print("\nPREDICTION BIAS")
print("-" * 70)

print(
    f"Mean prediction error : "
    f"₹{mean_prediction_error:.2f}"
)

if mean_prediction_error > 0:
    print(
        "The model tends to underpredict fares on average."
    )
elif mean_prediction_error < 0:
    print(
        "The model tends to overpredict fares on average."
    )
else:
    print(
        "The model has approximately zero average prediction bias."
    )


print("\nABSOLUTE ERROR DISTRIBUTION")
print("-" * 70)

print(
    f"Median absolute error : "
    f"₹{median_absolute_error:.2f}"
)

print(
    f"75th percentile      : "
    f"₹{percentile_75:.2f}"
)

print(
    f"90th percentile      : "
    f"₹{percentile_90:.2f}"
)

print(
    f"95th percentile      : "
    f"₹{percentile_95:.2f}"
)

print(
    f"99th percentile      : "
    f"₹{percentile_99:.2f}"
)


print("\nPROPORTION OF TEST PREDICTIONS")
print("-" * 70)

for threshold in [50, 100, 200, 300]:
    
    proportion = (
        error_analysis_df["absolute_error"]
        .le(threshold)
        .mean()
        * 100
    )
    
    print(
        f"≤ ₹{threshold:<3} : "
        f"{proportion:.2f}%"
    )

proportion_above_300 = (
    error_analysis_df["absolute_error"]
    .gt(300)
    .mean()
    * 100
)

print(
    f"> ₹300  : "
    f"{proportion_above_300:.2f}%"
)

FINAL MODEL — OVERALL ERROR ANALYSIS

PREDICTION BIAS
----------------------------------------------------------------------
Mean prediction error : ₹20.03
The model tends to underpredict fares on average.

ABSOLUTE ERROR DISTRIBUTION
----------------------------------------------------------------------
Median absolute error : ₹32.32
75th percentile      : ₹76.35
90th percentile      : ₹166.55
95th percentile      : ₹259.77
99th percentile      : ₹506.84

PROPORTION OF TEST PREDICTIONS
----------------------------------------------------------------------
≤ ₹50  : 62.99%
≤ ₹100 : 81.38%
≤ ₹200 : 92.30%
≤ ₹300 : 96.39%
> ₹300  : 3.61%


## Error Analysis by Ride Type

Different ride types have different pricing structures.

This analysis examines whether the final model's prediction errors vary substantially across ride types.

For each ride type, we will calculate:

- number of observations
- mean actual fare
- MAE
- RMSE
- mean prediction error

This will help identify ride types where the model consistently struggles.

In [111]:
# Error analysis by ride type

ride_error_analysis = (
    error_analysis_df
    .groupby("ride_name")
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=("error", lambda x: np.sqrt(np.mean(x ** 2))),
        mean_error=("error", "mean")
    )
    .sort_values("MAE", ascending=False)
)

ride_error_analysis.round(2)

,observations,mean_actual_fare,MAE,RMSE,mean_error
ride_name,,,,,
Black Rentals,2,1648.50,197.79,230.46,197.79
Innova,15,2442.49,163.09,302.54,152.06
XL,62,2216.07,155.60,222.27,113.33
XL Rentals,362,1249.26,128.51,228.29,41.35
Black,254,716.57,112.87,162.00,69.85
Premier Rentals,362,988.96,112.51,202.60,35.49
Sedan,62,1373.17,98.62,144.44,59.06
Go Rentals,362,775.59,90.68,158.42,-2.52
Hatchback,62,1195.39,88.28,125.62,42.75


## Error Analysis by Time Slot

Fare levels can vary across different collection time slots.

This analysis examines whether the final model performs differently across the six time slots.

For each time slot, we will calculate:

- number of observations
- mean actual fare
- MAE
- RMSE
- mean prediction error

This will help determine whether prediction errors are concentrated during particular periods of the day.

In [112]:
# Error analysis by time slot

time_error_analysis = (
    error_analysis_df
    .groupby("time_slot")
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=("error", lambda x: np.sqrt(np.mean(x ** 2))),
        mean_error=("error", "mean")
    )
    .sort_values("MAE", ascending=False)
)

time_error_analysis.round(2)

,observations,mean_actual_fare,MAE,RMSE,mean_error
time_slot,,,,,
18:00,1046,765.08,87.07,189.04,21.44
15:00,1021,825.95,85.82,154.74,37.96
00:00,2672,517.56,57.44,102.12,11.95
21:00,1248,594.77,57.06,92.49,21.46


## Error Analysis by Trip Distance

Trip distance is expected to have an important relationship with fare.

This analysis groups test observations into distance ranges and examines whether prediction error changes as trip distance increases.

For each distance group, we will calculate:

- number of observations
- mean actual fare
- MAE
- RMSE
- mean prediction error

This will help determine whether the model's errors are concentrated on short, medium, or long trips.

In [113]:
# Error analysis by trip distance

distance_bins = [
    -np.inf,
    5,
    10,
    20,
    30,
    50,
    np.inf
]

distance_labels = [
    "0-5 km",
    "5-10 km",
    "10-20 km",
    "20-30 km",
    "30-50 km",
    "50+ km"
]

error_analysis_df["distance_group"] = pd.cut(
    error_analysis_df["road_distance_km"],
    bins=distance_bins,
    labels=distance_labels
)

distance_error_analysis = (
    error_analysis_df
    .groupby("distance_group", observed=True)
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=("error", lambda x: np.sqrt(np.mean(x ** 2))),
        mean_error=("error", "mean")
    )
)

distance_error_analysis.round(2)

,observations,mean_actual_fare,MAE,RMSE,mean_error
distance_group,,,,,
0-5 km,226,332.58,32.22,58.29,15.73
5-10 km,1187,397.16,48.48,88.93,20.47
10-20 km,2963,565.79,58.14,99.96,8.35
20-30 km,1183,807.90,101.53,198.91,37.75
30-50 km,344,1234.06,91.95,159.98,36.31
50+ km,84,1969.73,173.23,243.90,120.67


## Analysis of the Largest Prediction Errors

Aggregate error analysis shows that prediction errors are concentrated in certain
ride types, time slots, and longer-distance trips.

The next step is to inspect the individual test observations with the largest
absolute prediction errors.

This can help identify whether large errors are associated with:

- unusually expensive fares
- long-distance trips
- particular ride types
- particular time slots
- specific routes
- combinations of these factors

These observations will help guide the next stage of model improvement.

In [114]:
# Largest individual prediction errors

largest_errors = (
    error_analysis_df
    .sort_values("absolute_error", ascending=False)
    [
        [
            "road_distance_km",
            "travel_time_minutes",
            "ride_name",
            "pickup",
            "dropoff",
            "time_slot",
            "day_of_week",
            "actual_fare",
            "predicted_fare",
            "error",
            "absolute_error"
        ]
    ]
    .head(20)
)

largest_errors.round(2)

,road_distance_km,travel_time_minutes,ride_name,pickup,dropoff,time_slot,day_of_week,actual_fare,predicted_fare,error,absolute_error
22722,26.14,61.58,XL Rentals,KLM Fashion Mall,Dasarahalli,18:00,Saturday,3706.80,1745.69,1961.11,1961.11
22720,26.14,61.58,XL Plus Rentals,KLM Fashion Mall,Dasarahalli,18:00,Saturday,3618.85,1802.81,1816.04,1816.04
22721,26.14,61.58,Premier Rentals,KLM Fashion Mall,Dasarahalli,18:00,Saturday,3198.00,1392.75,1805.25,1805.25
22719,26.14,61.58,Go Rentals,KLM Fashion Mall,Dasarahalli,18:00,Saturday,2558.00,1218.31,1339.69,1339.69
22724,26.14,61.58,Premier XL,KLM Fashion Mall,Dasarahalli,18:00,Saturday,2311.06,1016.44,1294.62,1294.62
22723,26.14,61.58,Uber Pet,KLM Fashion Mall,Dasarahalli,18:00,Saturday,2175.89,923.21,1252.68,1252.68
22718,26.14,61.58,UberXL,KLM Fashion Mall,Dasarahalli,18:00,Saturday,2070.50,895.75,1174.75,1174.75
22434,35.46,79.45,XL Rentals,Nexus Whitefield,Madavara,15:00,Saturday,3611.80,2613.57,998.23,998.23
22441,35.46,79.45,XL Plus Rentals,Nexus Whitefield,Madavara,15:00,Saturday,3523.85,2529.88,993.97,993.97
23070,21.87,58.59,XL Rentals,Brahmaputra Apartments,Amenities Hall,18:00,Saturday,2559.50,1590.51,968.99,968.99


## Investigating Feature Interactions

The largest individual prediction errors suggest that model performance may
depend on interactions between features rather than on individual features alone.

In particular, the largest errors contain repeated combinations of:

- specific ride types
- specific time slots
- specific days
- longer trip distances

The next analysis therefore examines model error across combinations of
`ride_name` and `time_slot`.

This will help determine whether certain ride types have systematically
different prediction behavior at particular times of day.

In [115]:
# Error analysis by ride type and time slot

ride_time_error_analysis = (
    error_analysis_df
    .groupby(["ride_name", "time_slot"])
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=("error", lambda x: np.sqrt(np.mean(x ** 2))),
        mean_error=("error", "mean")
    )
    .reset_index()
)

ride_time_error_analysis = (
    ride_time_error_analysis[
        ride_time_error_analysis["observations"] >= 20
    ]
    .sort_values("MAE", ascending=False)
)

ride_time_error_analysis.head(20).round(2)

,ride_name,time_slot,observations,mean_actual_fare,MAE,RMSE,mean_error
81,XL Rentals,15:00,62,1577.96,227.97,302.94,199.51
51,Premier Rentals,15:00,62,1270.81,196.16,266.90,173.07
82,XL Rentals,18:00,61,1408.46,173.43,337.69,74.75
52,Premier Rentals,18:00,61,1123.75,153.66,305.44,61.87
74,XL,18:00,27,2131.77,144.33,218.04,110.84
73,XL,15:00,24,2181.41,135.89,218.89,66.79
18,Black,21:00,71,757.31,131.86,170.78,114.15
37,Go Rentals,18:00,61,888.16,124.23,239.05,28.34
36,Go Rentals,15:00,62,1018.65,120.75,174.75,38.36
77,XL Plus Rentals,15:00,62,1570.32,118.74,235.74,103.96


## Investigating Prediction Error Relative to Trip Distance

Previous error analysis showed that prediction error increases substantially
for longer trips, particularly for trips above 20 km.

The ride type × time-slot analysis also showed that several rental ride types
have large errors at 15:00 and 18:00.

The next step is to examine the relationship between trip distance and
prediction error directly.

This will help determine whether the remaining errors are primarily associated
with longer trips and whether the model systematically underpredicts expensive
long-distance fares.

In [116]:
# Prediction error relative to trip distance

distance_error_analysis = (
    error_analysis_df
    .groupby(
        pd.cut(
            error_analysis_df["road_distance_km"],
            bins=[0, 5, 10, 20, 30, 50, np.inf],
            labels=["0-5 km", "5-10 km", "10-20 km",
                    "20-30 km", "30-50 km", "50+ km"],
            include_lowest=True
        )
    )
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=("error", lambda x: np.sqrt(np.mean(x ** 2))),
        mean_error=("error", "mean")
    )
    .round(2)
)

distance_error_analysis

,observations,mean_actual_fare,MAE,RMSE,mean_error
road_distance_km,,,,,
0-5 km,226,332.58,32.22,58.29,15.73
5-10 km,1187,397.16,48.48,88.93,20.47
10-20 km,2963,565.79,58.14,99.96,8.35
20-30 km,1183,807.90,101.53,198.91,37.75
30-50 km,344,1234.06,91.95,159.98,36.31
50+ km,84,1969.73,173.23,243.90,120.67


## High-Fare Prediction Analysis

The error analysis shows that the model has difficulty with longer trips and
with several high-priced rental and premium ride observations.

The largest individual errors are predominantly underpredictions, where the
actual fare is substantially higher than the predicted fare.

The next step is to examine model performance across different fare ranges.

This will determine whether the model systematically loses accuracy as the
actual fare increases and whether the remaining errors are concentrated in
the high-fare tail.

In [117]:
# Error analysis by actual fare range

fare_bins = [0, 300, 500, 800, 1200, 1600, 2000, 3000, np.inf]

fare_labels = [
    "₹0–300",
    "₹300–500",
    "₹500–800",
    "₹800–1200",
    "₹1200–1600",
    "₹1600–2000",
    "₹2000–3000",
    "₹3000+"
]

fare_error_analysis = (
    error_analysis_df
    .assign(
        fare_group=pd.cut(
            error_analysis_df["actual_fare"],
            bins=fare_bins,
            labels=fare_labels,
            include_lowest=True
        )
    )
    .groupby("fare_group", observed=False)
    .agg(
        observations=("actual_fare", "size"),
        mean_actual_fare=("actual_fare", "mean"),
        mean_predicted_fare=("predicted_fare", "mean"),
        MAE=("absolute_error", "mean"),
        RMSE=("error", lambda x: np.sqrt(np.mean(x ** 2))),
        mean_error=("error", "mean")
    )
)

fare_error_analysis.round(2)

,observations,mean_actual_fare,mean_predicted_fare,MAE,RMSE,mean_error
fare_group,,,,,,
₹0–300,1652,183.95,187.69,26.30,37.49,-3.74
₹300–500,1274,396.79,393.81,51.56,71.53,2.98
₹500–800,1422,635.92,626.76,68.31,101.61,9.16
₹800–1200,789,973.51,943.63,98.48,149.41,29.88
₹1200–1600,623,1295.50,1254.02,84.58,159.60,41.47
₹1600–2000,124,1791.26,1598.45,233.48,316.87,192.81
₹2000–3000,89,2361.10,2093.07,288.70,420.00,268.03
₹3000+,14,3402.37,2540.19,862.18,1042.37,862.18


## Model Improvement — Gradient Boosting Regressor

The error analysis shows that the Random Forest performs well for the majority
of observations but systematically underpredicts higher-priced fares.

The largest errors are concentrated in:

- high-fare observations
- longer trips
- rental and premium ride types
- certain ride type × time-slot combinations

The log-target Random Forest produced only a modest improvement over the
original Random Forest.

The next model evaluated will therefore be a Gradient Boosting Regressor.

Gradient Boosting is a sequential ensemble method in which each new tree
attempts to correct errors made by the existing ensemble. This makes it a
useful alternative to Random Forest for determining whether the remaining
systematic errors can be reduced.

The same development data, date-based cross-validation folds, preprocessing
strategy, and untouched test set will be used so that the comparison with the
Random Forest remains fair.

In [118]:
# Create Gradient Boosting preprocessing and model pipeline

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline


# Numerical features
numerical_features = [
    "road_distance_km",
    "travel_time_minutes"
]


# Categorical features
categorical_features = [
    "ride_name",
    "pickup_category",
    "drop_category",
    "time_slot",
    "day_of_week",
    "pickup",
    "dropoff"
]


# Preprocessing
gb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)


# Gradient Boosting model
gb_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=2,
    random_state=42
)


# Complete pipeline
gb_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            gb_preprocessor
        ),
        (
            "model",
            gb_model
        )
    ]
)


print("Gradient Boosting pipeline created successfully.")

Gradient Boosting pipeline created successfully.


In [119]:
# Evaluate Gradient Boosting using the same date-based CV folds

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


gb_cv_results = []

print("Evaluating Gradient Boosting...")
print(f"Number of CV folds: {len(date_cv_splits)}")

for fold_number, (train_idx, validation_idx) in enumerate(
    date_cv_splits,
    start=1
):

    # Create a fresh copy of the pipeline for this fold
    fold_model = clone(gb_pipeline)

    # Split development data
    X_train_fold = X_tuning.iloc[train_idx]
    y_train_fold = y_tuning.iloc[train_idx]

    X_validation_fold = X_tuning.iloc[validation_idx]
    y_validation_fold = y_tuning.iloc[validation_idx]

    # Train
    fold_model.fit(
        X_train_fold,
        y_train_fold
    )

    # Predict
    validation_predictions = fold_model.predict(
        X_validation_fold
    )

    # Calculate metrics
    fold_mae = mean_absolute_error(
        y_validation_fold,
        validation_predictions
    )

    fold_rmse = np.sqrt(
        mean_squared_error(
            y_validation_fold,
            validation_predictions
        )
    )

    fold_r2 = r2_score(
        y_validation_fold,
        validation_predictions
    )

    gb_cv_results.append(
        {
            "fold": fold_number,
            "MAE": fold_mae,
            "RMSE": fold_rmse,
            "R2": fold_r2
        }
    )

    print(
        f"Fold {fold_number}: "
        f"MAE = ₹{fold_mae:.2f}, "
        f"RMSE = ₹{fold_rmse:.2f}, "
        f"R² = {fold_r2:.4f}"
    )


# Convert results to DataFrame
gb_cv_results_df = pd.DataFrame(
    gb_cv_results
)


# Calculate mean and standard deviation
gb_mean_mae = gb_cv_results_df["MAE"].mean()
gb_mean_rmse = gb_cv_results_df["RMSE"].mean()
gb_mean_r2 = gb_cv_results_df["R2"].mean()

gb_std_mae = gb_cv_results_df["MAE"].std()
gb_std_rmse = gb_cv_results_df["RMSE"].std()
gb_std_r2 = gb_cv_results_df["R2"].std()


print("\n" + "=" * 70)
print("GRADIENT BOOSTING — DATE-BASED CROSS-VALIDATION")
print("=" * 70)

print(f"\nMean MAE  : ₹{gb_mean_mae:.2f}")
print(f"Mean RMSE : ₹{gb_mean_rmse:.2f}")
print(f"Mean R²   : {gb_mean_r2:.4f}")

print("\nStandard deviation:")
print(f"MAE  : ₹{gb_std_mae:.2f}")
print(f"RMSE : ₹{gb_std_rmse:.2f}")
print(f"R²   : {gb_std_r2:.4f}")

Evaluating Gradient Boosting...
Number of CV folds: 4
Fold 1: MAE = ₹123.78, RMSE = ₹212.34, R² = 0.7819
Fold 2: MAE = ₹118.10, RMSE = ₹187.24, R² = 0.8309
Fold 3: MAE = ₹104.48, RMSE = ₹167.44, R² = 0.8494
Fold 4: MAE = ₹101.63, RMSE = ₹148.97, R² = 0.9160

GRADIENT BOOSTING — DATE-BASED CROSS-VALIDATION

Mean MAE  : ₹112.00
Mean RMSE : ₹179.00
Mean R²   : 0.8446

Standard deviation:
MAE  : ₹10.65
RMSE : ₹27.17
R²   : 0.0555


### Model Comparison

Three Random Forest / Gradient Boosting approaches were evaluated using the same
date-based cross-validation strategy:

| Model | Mean MAE | Mean RMSE | Mean R² |
|---|---:|---:|---:|
| Random Forest | ₹100.16 | ₹178.03 | 0.8450 |
| Log-target Random Forest | ₹97.79 | ₹172.63 | 0.8536 |
| Gradient Boosting | ₹112.00 | ₹179.00 | 0.8446 |

The log-target Random Forest achieved the lowest cross-validation MAE and RMSE
and the highest mean R² among the evaluated models.

Therefore, the log-target Random Forest is selected as the final model based on
date-based cross-validation performance.

The test set is kept separate from model selection and is used only to provide
the final held-out performance estimate.

In [120]:
# Feature importance analysis for the final log-target Random Forest

import pandas as pd
import matplotlib.pyplot as plt


# Extract the fitted preprocessing step
preprocessor = final_log_rf_pipeline.named_steps["preprocessor"]

# Extract the fitted Random Forest
final_rf_model = final_log_rf_pipeline.named_steps["model"]


# Get feature names after preprocessing
transformed_feature_names = (
    preprocessor.get_feature_names_out()
)

# Get Random Forest feature importances
feature_importances = final_rf_model.feature_importances_


# Create a DataFrame
feature_importance_df = pd.DataFrame({
    "transformed_feature": transformed_feature_names,
    "importance": feature_importances
})


# Sort by importance
feature_importance_df = (
    feature_importance_df
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 70)
print("TOP 20 TRANSFORMED FEATURES")
print("=" * 70)

display(
    feature_importance_df.head(20)
)

TOP 20 TRANSFORMED FEATURES


,transformed_feature,importance
0,num__road_distance_km,0.248734
1,cat__ride_name_Bike,0.108648
2,cat__ride_name_Bike Saver,0.106085
3,cat__ride_name_XL Plus Rentals,0.099847
4,cat__ride_name_Go Non AC,0.068776
5,cat__ride_name_Auto,0.061469
6,cat__ride_name_Uber Go AC,0.055852
7,cat__ride_name_XL Rentals,0.052842
8,num__travel_time_minutes,0.041208
9,cat__ride_name_Premier Rentals,0.023151


In [121]:
# Aggregate transformed feature importances back to original features

feature_importance_df["original_feature"] = (
    feature_importance_df["transformed_feature"]
    .str.replace(
        r"^(num__|cat__)",
        "",
        regex=True
    )
    .str.split("_", n=1)
    .str[0]
)

# The above approach does not correctly identify multi-word feature names,
# so use the original feature list to map them properly.

original_features = [
    "road_distance_km",
    "travel_time_minutes",
    "ride_name",
    "pickup_category",
    "drop_category",
    "time_slot",
    "day_of_week",
    "pickup",
    "dropoff"
]


def get_original_feature(transformed_name):

    # Numerical features
    if transformed_name.startswith("num__"):
        return transformed_name.replace(
            "num__",
            ""
        )

    # Categorical features
    if transformed_name.startswith("cat__"):

        categorical_name = transformed_name.replace(
            "cat__",
            "",
            1
        )

        for feature in original_features:
            if categorical_name.startswith(feature + "_"):
                return feature

            if categorical_name == feature:
                return feature

    return transformed_name


feature_importance_df["original_feature"] = (
    feature_importance_df["transformed_feature"]
    .apply(get_original_feature)
)


# Aggregate importance
original_feature_importance = (
    feature_importance_df
    .groupby("original_feature")["importance"]
    .sum()
    .sort_values(
        ascending=False
    )
    .reset_index()
)


print("=" * 70)
print("FEATURE IMPORTANCE — ORIGINAL FEATURES")
print("=" * 70)

display(
    original_feature_importance
)

FEATURE IMPORTANCE — ORIGINAL FEATURES


,original_feature,importance
0,ride_name,0.638690
1,road_distance_km,0.248734
2,travel_time_minutes,0.041208
3,time_slot,0.022390
4,pickup,0.016452
5,dropoff,0.014511
6,day_of_week,0.008995
7,pickup_category,0.005126
8,drop_category,0.003894


## Feature Reduction Experiment

Feature importance analysis showed that `ride_name` and `road_distance_km`
account for most of the model's feature importance.

The remaining features have smaller individual importance values.

However, feature importance alone is not sufficient reason to remove features.
Therefore, a controlled feature-reduction experiment will be performed.

The reduced feature set will remove:

- `day_of_week`
- `pickup_category`
- `drop_category`

while retaining:

- `road_distance_km`
- `travel_time_minutes`
- `ride_name`
- `time_slot`
- `pickup`
- `dropoff`

The reduced model will be evaluated using the same date-based cross-validation
splits used for the final model.

The original 9-feature model will remain the reference model.

In [122]:
# Create reduced-feature dataset and log-target Random Forest pipeline

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor


# ------------------------------------------------------------
# 1. Reduced feature list
# ------------------------------------------------------------

reduced_features = [
    "road_distance_km",
    "travel_time_minutes",
    "ride_name",
    "time_slot",
    "pickup",
    "dropoff"
]


# Create reduced development dataset
X_tuning_reduced = X_tuning[
    reduced_features
].copy()


# ------------------------------------------------------------
# 2. Define numerical and categorical features
# ------------------------------------------------------------

reduced_numerical_features = [
    "road_distance_km",
    "travel_time_minutes"
]

reduced_categorical_features = [
    "ride_name",
    "time_slot",
    "pickup",
    "dropoff"
]


# ------------------------------------------------------------
# 3. Preprocessing
# ------------------------------------------------------------

reduced_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            reduced_numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            reduced_categorical_features
        )
    ]
)


# ------------------------------------------------------------
# 4. Random Forest
# ------------------------------------------------------------

reduced_rf = RandomForestRegressor(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=1,
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)


# ------------------------------------------------------------
# 5. Base pipeline
# ------------------------------------------------------------

reduced_base_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            reduced_preprocessor
        ),
        (
            "model",
            reduced_rf
        )
    ]
)


# ------------------------------------------------------------
# 6. Log-target transformation
# ------------------------------------------------------------

reduced_log_rf_pipeline = TransformedTargetRegressor(
    regressor=reduced_base_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)


print("Reduced-feature log-target Random Forest pipeline created successfully.")

print("\nOriginal number of features:", len(X_tuning.columns))
print("Reduced number of features:", len(reduced_features))

print("\nReduced features:")
print(reduced_features)

Reduced-feature log-target Random Forest pipeline created successfully.

Original number of features: 9
Reduced number of features: 6

Reduced features:
['road_distance_km', 'travel_time_minutes', 'ride_name', 'time_slot', 'pickup', 'dropoff']


In [123]:
# Evaluate reduced-feature log-target Random Forest
# using the same date-based CV folds

from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd


reduced_cv_results = []

print("Evaluating reduced-feature log-target Random Forest...")
print(f"Number of CV folds: {len(date_cv_splits)}")


for fold_number, (train_idx, validation_idx) in enumerate(
    date_cv_splits,
    start=1
):

    # Fresh copy for this fold
    fold_model = clone(
        reduced_log_rf_pipeline
    )

    # Split development data
    X_train_fold = X_tuning_reduced.iloc[train_idx]
    y_train_fold = y_tuning.iloc[train_idx]

    X_validation_fold = X_tuning_reduced.iloc[validation_idx]
    y_validation_fold = y_tuning.iloc[validation_idx]

    # Train
    fold_model.fit(
        X_train_fold,
        y_train_fold
    )

    # Predict
    validation_predictions = fold_model.predict(
        X_validation_fold
    )

    # Metrics
    fold_mae = mean_absolute_error(
        y_validation_fold,
        validation_predictions
    )

    fold_rmse = np.sqrt(
        mean_squared_error(
            y_validation_fold,
            validation_predictions
        )
    )

    fold_r2 = r2_score(
        y_validation_fold,
        validation_predictions
    )

    reduced_cv_results.append(
        {
            "fold": fold_number,
            "MAE": fold_mae,
            "RMSE": fold_rmse,
            "R2": fold_r2
        }
    )

    print(
        f"Fold {fold_number}: "
        f"MAE = ₹{fold_mae:.2f}, "
        f"RMSE = ₹{fold_rmse:.2f}, "
        f"R² = {fold_r2:.4f}"
    )


# Convert results to DataFrame
reduced_cv_results_df = pd.DataFrame(
    reduced_cv_results
)


# Mean performance
reduced_mean_mae = reduced_cv_results_df["MAE"].mean()
reduced_mean_rmse = reduced_cv_results_df["RMSE"].mean()
reduced_mean_r2 = reduced_cv_results_df["R2"].mean()


# Standard deviation
reduced_std_mae = reduced_cv_results_df["MAE"].std()
reduced_std_rmse = reduced_cv_results_df["RMSE"].std()
reduced_std_r2 = reduced_cv_results_df["R2"].std()


print("\n" + "=" * 70)
print("REDUCED-FEATURE LOG-TARGET RANDOM FOREST")
print("DATE-BASED CROSS-VALIDATION")
print("=" * 70)

print(f"\nMean MAE  : ₹{reduced_mean_mae:.2f}")
print(f"Mean RMSE : ₹{reduced_mean_rmse:.2f}")
print(f"Mean R²   : {reduced_mean_r2:.4f}")

print("\nStandard deviation:")
print(f"MAE  : ₹{reduced_std_mae:.2f}")
print(f"RMSE : ₹{reduced_std_rmse:.2f}")
print(f"R²   : {reduced_std_r2:.4f}")

Evaluating reduced-feature log-target Random Forest...
Number of CV folds: 4
Fold 1: MAE = ₹116.58, RMSE = ₹212.84, R² = 0.7809
Fold 2: MAE = ₹105.25, RMSE = ₹184.17, R² = 0.8364
Fold 3: MAE = ₹88.79, RMSE = ₹163.29, R² = 0.8568
Fold 4: MAE = ₹81.34, RMSE = ₹136.40, R² = 0.9296

REDUCED-FEATURE LOG-TARGET RANDOM FOREST
DATE-BASED CROSS-VALIDATION

Mean MAE  : ₹97.99
Mean RMSE : ₹174.18
Mean R²   : 0.8509

Standard deviation:
MAE  : ₹15.92
RMSE : ₹32.35
R²   : 0.0615


## Feature-Reduction Decision

The reduced-feature log-target Random Forest was evaluated using the same
date-based cross-validation splits as the original 9-feature model.

The reduced model achieved:

- Mean MAE: ₹97.99
- Mean RMSE: ₹174.18
- Mean R²: 0.8509

The original 9-feature model achieved:

- Mean MAE: ₹97.79
- Mean RMSE: ₹172.63
- Mean R²: 0.8536

The reduced model therefore performed slightly worse across all three
evaluation metrics.

Although `pickup_category`, `drop_category`, and `day_of_week` had relatively
low aggregate feature importance, removing them resulted in a small
performance degradation.

Therefore, the original 9-feature representation will be retained for the
final model.

The next improvement stage will focus on understanding and reducing the
remaining prediction errors, particularly for expensive fares, long-distance
trips, and rental ride types.

## Hyperparameter Tuning — Log-Target Random Forest

The original 9-feature log-target Random Forest is retained as the base model.

The next experiment is hyperparameter tuning. The objective is to determine
whether the Random Forest can improve its generalization performance through
changes to parameters such as:

- number of trees (`n_estimators`)
- maximum tree depth (`max_depth`)
- minimum samples required at a leaf (`min_samples_leaf`)
- number of features considered at each split (`max_features`)

The same date-based cross-validation strategy will be used so that the
hyperparameter search does not use the final test set.

The final test set will remain untouched until the best configuration has
been selected.

In [124]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
import numpy as np


# Base Random Forest
tuned_rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)


# Hyperparameter search space
param_distributions = {
    "model__n_estimators": [300, 500, 700, 1000],
    "model__max_depth": [None, 15, 20, 25, 30],
    "model__min_samples_leaf": [1, 2, 4, 6],
    "model__max_features": [0.5, 0.7, 1.0]
}


# Use the original 9-feature preprocessing pipeline
tuned_rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", tuned_rf)
    ]
)


print("Hyperparameter tuning pipeline created successfully.")

Hyperparameter tuning pipeline created successfully.


In [128]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np


# ------------------------------------------------------------
# Randomized hyperparameter search
# ------------------------------------------------------------

random_search = RandomizedSearchCV(
    estimator=tuned_rf_pipeline,
    param_distributions=param_distributions,
    n_iter=15,
    scoring="neg_mean_absolute_error",
    cv=date_cv_splits,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)


print("Starting hyperparameter tuning...")
print("Number of configurations tested:", 15)
print("Number of CV folds:", len(date_cv_splits))
print(
    "Total model fits:",
    15 * len(date_cv_splits)
)


random_search.fit(
    X_tuning,
    y_tuning
)


print("\nHyperparameter tuning completed successfully.")

Starting hyperparameter tuning...
Number of configurations tested: 15
Number of CV folds: 4
Total model fits: 60
Fitting 4 folds for each of 15 candidates, totalling 60 fits

Hyperparameter tuning completed successfully.


In [129]:
# Display the best hyperparameter configuration

print("=" * 70)
print("BEST HYPERPARAMETER CONFIGURATION")
print("=" * 70)

print("\nBest CV MAE:")
print(f"₹{-random_search.best_score_:.2f}")

print("\nBest hyperparameters:")

for parameter, value in random_search.best_params_.items():
    print(f"{parameter}: {value}")

BEST HYPERPARAMETER CONFIGURATION

Best CV MAE:
₹103.37

Best hyperparameters:
model__n_estimators: 700
model__min_samples_leaf: 1
model__max_features: 0.7
model__max_depth: 20


## Final Tuned Random Forest

The best hyperparameters identified through date-based cross-validation are now used to create the final Random Forest pipeline.

The final model will be trained on the complete training set and evaluated once on the untouched test set.

The test set is not used during hyperparameter tuning.

In [130]:
# ============================================================
# FINAL TUNED RANDOM FOREST
# ============================================================

best_rf_pipeline = random_search.best_estimator_

print("Final tuned Random Forest pipeline created successfully.")

print("\nBest hyperparameters:")
for parameter, value in random_search.best_params_.items():
    print(f"{parameter}: {value}")

Final tuned Random Forest pipeline created successfully.

Best hyperparameters:
model__n_estimators: 700
model__min_samples_leaf: 1
model__max_features: 0.7
model__max_depth: 20


In [132]:
# ============================================================
# CREATE FINAL TUNED REDUCED-FEATURE RANDOM FOREST PIPELINE
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Reduced feature groups
reduced_numeric_features = [
    "road_distance_km",
    "travel_time_minutes"
]

reduced_categorical_features = [
    "ride_name",
    "time_slot",
    "pickup",
    "dropoff"
]

# Preprocessing for reduced feature set
reduced_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            reduced_numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            reduced_categorical_features
        )
    ]
)

# Final Random Forest with best hyperparameters
final_tuned_reduced_rf_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            reduced_preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=700,
                min_samples_leaf=1,
                max_features=0.7,
                max_depth=20,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("Final tuned reduced-feature Random Forest pipeline created successfully.")

Final tuned reduced-feature Random Forest pipeline created successfully.


In [133]:
# ============================================================
# TRAIN FINAL TUNED REDUCED-FEATURE RANDOM FOREST
# ============================================================

print("Training final tuned reduced-feature Random Forest...")

final_tuned_reduced_rf_pipeline.fit(
    X_tuning_reduced,
    y_tuning
)

print("Final tuned reduced-feature Random Forest trained successfully.")

Training final tuned reduced-feature Random Forest...
Final tuned reduced-feature Random Forest trained successfully.


In [135]:
# Select the same reduced features used by the final tuned model
X_test_reduced = X_test[reduced_features].copy()

print("X_test_reduced shape:", X_test_reduced.shape)
print("Features:")
print(X_test_reduced.columns.tolist())

X_test_reduced shape: (5987, 6)
Features:
['road_distance_km', 'travel_time_minutes', 'ride_name', 'time_slot', 'pickup', 'dropoff']


In [137]:
print("Log prediction statistics:")
print("Minimum:", np.min(y_test_pred_log))
print("Maximum:", np.max(y_test_pred_log))
print("Mean   :", np.mean(y_test_pred_log))
print("NaN count:", np.isnan(y_test_pred_log).sum())
print("Inf count:", np.isinf(y_test_pred_log).sum())

Log prediction statistics:
Minimum: 33.34675417888915
Maximum: 3245.087626734687
Mean   : 617.5036388520465
NaN count: 0
Inf count: 0


In [138]:
print("y_tuning statistics:")
print("Minimum:", y_tuning.min())
print("Maximum:", y_tuning.max())
print("Mean   :", y_tuning.mean())
print("Median :", y_tuning.median())

y_tuning statistics:
Minimum: 4.08
Maximum: 3611.8
Mean   : 632.7405977043674
Median : 514.545


In [139]:
print("y_test statistics:")
print("Minimum:", y_test.min())
print("Maximum:", y_test.max())
print("Mean   :", y_test.mean())
print("Median :", y_test.median())

y_test statistics:
Minimum: 10.12
Maximum: 3987.61
Mean   : 629.4903824954068
Median : 513.31


In [140]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("Evaluating final tuned reduced-feature Random Forest on test set...")

# Predictions are already on the original ₹ scale
y_test_pred = final_tuned_reduced_rf_pipeline.predict(X_test_reduced)

# Evaluate
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print("\nFinal Tuned Reduced-Feature Random Forest — Test Results")
print("-" * 60)
print(f"MAE  : ₹{test_mae:.2f}")
print(f"RMSE : ₹{test_rmse:.2f}")
print(f"R²   : {test_r2:.4f}")

Evaluating final tuned reduced-feature Random Forest on test set...

Final Tuned Reduced-Feature Random Forest — Test Results
------------------------------------------------------------
MAE  : ₹71.32
RMSE : ₹132.13
R²   : 0.9232


In [141]:
import pandas as pd

model_comparison = pd.DataFrame({
    "Model": [
        "Log-target Random Forest (9 features)",
        "Tuned Random Forest (6 features)"
    ],
    "Features": [
        9,
        6
    ],
    "MAE": [
        67.38,
        71.32
    ],
    "RMSE": [
        129.48,
        132.13
    ],
    "R2": [
        0.9262,
        0.9232
    ]
})

model_comparison

,Model,Features,MAE,RMSE,R2
0,Log-target Random Forest (9 features),9,67.38,129.48,0.9262
1,Tuned Random Forest (6 features),6,71.32,132.13,0.9232


In [142]:
print("9-feature log-target model:")
print(final_rf_pipeline)

9-feature log-target model:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['road_distance_km',
                                                   'travel_time_minutes']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['ride_name',
                                                   'pickup_category',
                                                   'drop_category', 'time_slot',
                                                   'day_of_week', 'pickup',
                                                   'dropoff'])])),
                ('model',
                 RandomForestRegressor(n_estimators=600, n_jobs=-1,
                                       random_state=42))])


In [144]:
import joblib

model_path = "../models/final_fare_prediction_model.joblib"

joblib.dump(final_rf_pipeline, model_path)

print(f"Final model saved successfully at: {model_path}")

Final model saved successfully at: ../models/final_fare_prediction_model.joblib


In [145]:
import joblib
import numpy as np

# Load the saved model
loaded_model = joblib.load("../models/final_fare_prediction_model.joblib")

# Take one test observation
sample = X_test.iloc[[0]]

# Predict in log scale
sample_pred_log = loaded_model.predict(sample)

# Convert back to ₹
sample_pred = np.expm1(sample_pred_log)

print("Model loaded successfully.")
print(f"Actual fare     : ₹{y_test.iloc[0]:.2f}")
print(f"Predicted fare  : ₹{sample_pred[0]:.2f}")

Model loaded successfully.
Actual fare     : ₹490.12
Predicted fare  : ₹601151593347660275348020157335232668916948740313455012088728578089039512299723970185053563166908249294607467366677805651500281155753559192235688534893120528261990776258597921117752382627066113168895989378294218752.00


In [146]:
# Compare the saved model's raw prediction with the in-memory model

sample = X_test.iloc[[0]]

loaded_raw_prediction = loaded_model.predict(sample)
memory_raw_prediction = final_rf_pipeline.predict(sample)

print("Raw prediction from loaded model:")
print(loaded_raw_prediction[0])

print("\nRaw prediction from in-memory final_rf_pipeline:")
print(memory_raw_prediction[0])

Raw prediction from loaded model:
489.9417166666663

Raw prediction from in-memory final_rf_pipeline:
489.9417166666664


In [147]:
# Show model-related variables currently available in the notebook

model_variables = [
    name for name in globals()
    if any(keyword in name.lower() for keyword in [
        "pipeline",
        "rf",
        "random",
        "model"
    ])
]

print("Model-related variables currently available:")
for name in sorted(model_variables):
    print("-", name)

Model-related variables currently available:
- Pipeline
- RandomForestRegressor
- RandomizedSearchCV
- baseline_pipeline
- best_rf_pipeline
- final_log_rf_pipeline
- final_rf_model
- final_rf_params
- final_rf_pipeline
- final_route_rf_pipeline
- final_tuned_reduced_rf_pipeline
- fold_model
- fold_pipeline
- gb_model
- gb_pipeline
- gradient_boosting_pipeline
- loaded_model
- log_rf_cv_results
- log_rf_cv_results_df
- log_rf_pipeline
- log_rf_preprocessor
- model_comparison
- model_comparison_sorted
- model_path
- random_forest_pipeline
- random_search
- reduced_base_pipeline
- reduced_log_rf_pipeline
- reduced_rf
- rf_cv_results
- rf_grid_search
- rf_parameter_distributions
- rf_random_search
- rf_targeted_parameters
- rf_targeted_results
- rf_tuning_results
- route_rf_pipeline
- tuned_gradient_boosting_pipeline
- tuned_random_forest_pipeline
- tuned_rf
- tuned_rf_cv_results
- tuned_rf_pipeline


In [148]:
import numpy as np

sample = X_test.iloc[[0]]

# Raw prediction from the suspected log-target model
log_prediction = final_log_rf_pipeline.predict(sample)

print("Raw prediction from final_log_rf_pipeline:")
print(log_prediction[0])

print("\nAfter converting back from log1p:")
print(np.expm1(log_prediction[0]))

print("\nActual fare:")
print(y_test.iloc[0])

Raw prediction from final_log_rf_pipeline:
6.191756746562423

After converting back from log1p:
487.70388141073533

Actual fare:
490.12


In [149]:
import joblib

model_path = "../models/final_fare_prediction_model.joblib"

joblib.dump(final_log_rf_pipeline, model_path)

print(f"Correct log-target model saved successfully at: {model_path}")

Correct log-target model saved successfully at: ../models/final_fare_prediction_model.joblib


In [150]:
import joblib
import numpy as np

# Load the final saved model
final_saved_model = joblib.load(
    "../models/final_fare_prediction_model.joblib"
)

# Use the same test observation
sample = X_test.iloc[[0]]

# Predict in log1p scale
prediction_log = final_saved_model.predict(sample)

# Convert back to ₹
prediction_fare = np.expm1(prediction_log[0])

print("Final saved model loaded successfully.")
print(f"Actual fare    : ₹{y_test.iloc[0]:.2f}")
print(f"Predicted fare : ₹{prediction_fare:.2f}")

Final saved model loaded successfully.
Actual fare    : ₹490.12
Predicted fare : ₹487.70


In [151]:
import joblib
import numpy as np
import pandas as pd

# Load the final saved model
final_model = joblib.load(
    "../models/final_fare_prediction_model.joblib"
)


def predict_fare(
    road_distance_km,
    travel_time_minutes,
    ride_name,
    pickup_category,
    drop_category,
    time_slot,
    day_of_week,
    pickup,
    dropoff
):
    """
    Predict Uber fare for a single trip.
    """

    # Create input DataFrame with the exact feature names
    input_data = pd.DataFrame([{
        "road_distance_km": road_distance_km,
        "travel_time_minutes": travel_time_minutes,
        "ride_name": ride_name,
        "pickup_category": pickup_category,
        "drop_category": drop_category,
        "time_slot": time_slot,
        "day_of_week": day_of_week,
        "pickup": pickup,
        "dropoff": dropoff
    }])

    # Model predicts log1p(fare)
    predicted_log_fare = final_model.predict(input_data)

    # Convert back to original ₹ scale
    predicted_fare = np.expm1(predicted_log_fare[0])

    return predicted_fare


print("Prediction function created successfully.")

Prediction function created successfully.


In [152]:
# Test the prediction function using the first test observation

sample = X_test.iloc[0]

predicted_fare = predict_fare(
    road_distance_km=sample["road_distance_km"],
    travel_time_minutes=sample["travel_time_minutes"],
    ride_name=sample["ride_name"],
    pickup_category=sample["pickup_category"],
    drop_category=sample["drop_category"],
    time_slot=sample["time_slot"],
    day_of_week=sample["day_of_week"],
    pickup=sample["pickup"],
    dropoff=sample["dropoff"]
)

print(f"Actual fare    : ₹{y_test.iloc[0]:.2f}")
print(f"Predicted fare : ₹{predicted_fare:.2f}")

Actual fare    : ₹490.12
Predicted fare : ₹487.70


In [153]:
predicted_fare = predict_fare(
    road_distance_km=12.5,
    travel_time_minutes=35,
    ride_name="Uber Go AC",
    pickup_category="metro_station",
    drop_category="mall",
    time_slot="18:00",
    day_of_week="Wednesday",
    pickup="Indiranagar",
    dropoff="Nexus Koramangala"
)

print(f"Predicted Uber fare: ₹{predicted_fare:.2f}")

Predicted Uber fare: ₹325.35


In [154]:
print("Pickup locations:", df["pickup"].nunique())
print("Dropoff locations:", df["dropoff"].nunique())
print("Ride types:", df["ride_name"].nunique())
print("Pickup categories:", df["pickup_category"].nunique())
print("Drop categories:", df["drop_category"].nunique())
print("Time slots:", df["time_slot"].unique())
print("Days:", df["day_of_week"].unique())

Pickup locations: 236
Dropoff locations: 229
Ride types: 25
Pickup categories: 11
Drop categories: 10
Time slots: <StringArray>
['15:00', '00:00', '18:00', '21:00', '09:00', '12:00']
Length: 6, dtype: str
Days: <StringArray>
['Sunday', 'Wednesday', 'Saturday', 'Monday', 'Tuesday', 'Thursday', 'Friday']
Length: 7, dtype: str
